# LeetCode 4091 — Count Magical Substring Pairs II (Hard)

Notebook này dạy bài **Count Magical Substring Pairs II** (Biweekly Contest 194, câu 4, độ khó Hard) từ con số 0, dành cho người mới.

Skeleton trên LeetCode:

```python
class Solution:
    def countMagicalPairs(self, s: str) -> int:
        ...
```

Nhìn qua thì bài chỉ là "đếm bộ ba chỉ số". Nhưng với `n = 10^5`, số bộ ba lên tới khoảng `1.7 · 10^14` — không thể thử từng cái. Ta sẽ đi từ cách ngây thơ nhất, rồi từng bước khám phá **3 quan sát** giúp bài toán chỉ còn khoảng `26 · n` phép tính.

---

## Sau notebook này, bạn sẽ nắm được

1. Đọc đúng đề: bộ ba `(i, j, k)` cắt đoạn `s[i..k]` thành **hai mảnh kề nhau** `s1 | s2`.
2. Kỹ thuật **cố định vết cắt** `j` cùng **nguyên lý nhân** để biến "đếm bộ ba" thành "đếm hai phía rồi nhân lại".
3. Tính chất **đơn điệu** (monotonic): kéo dài một substring thì số ký tự phân biệt không bao giờ giảm.
4. Ý tưởng **điểm gãy** (breakpoint): với 26 chữ cái, số ký tự phân biệt của các hậu tố chỉ "nhảy bậc" tối đa 26 lần, đúng tại các vị trí **xuất hiện cuối cùng** (last occurrence).
5. Mẹo kinh điển **"đúng d = nhiều nhất d − nhiều nhất (d − 1)"** cùng kỹ thuật **cửa sổ trượt / hai con trỏ** (sliding window / two pointers).
6. Cách viết `class Solution` chạy nhanh trong Python, kiểm thử chéo với brute force, và tránh các lỗi off-by-one rất hay gặp.
7. Nhận diện dạng bài "đếm cặp đoạn kề nhau theo số ký tự phân biệt" để tự giải các bài tương tự.

## Các cách giải sẽ học

| Cách | Ý tưởng chính | Thời gian |
|---|---|---|
| Cách 1 | Brute Force: thử mọi bộ ba `(i, j, k)`, đếm bằng `set` | `O(n^4)` |
| Cách 2 | Cố định vết cắt `j`, đếm hai phía rồi nhân | `O(n^2)` |
| Cách 3 | Điểm gãy theo vị trí xuất hiện cuối / đầu tiên | `O(26 · n)` (kèm sắp xếp ≤ 26 số mỗi bước) |
| Cách 4 | Cửa sổ trượt "nhiều nhất d" cho từng `d = 1..26` | `O(26 · n)` |

Cách 3 (bản tinh chỉnh, không cần sắp xếp) sẽ là bản nộp LeetCode. Cách 4 là lời giải thay thế dùng một pattern cực kỳ đáng học.

# 1. Đề bài

Cho chuỗi `s` có độ dài `n`, chỉ gồm chữ cái tiếng Anh thường (`'a'` … `'z'`).

Với **mọi bộ ba chỉ số** `(i, j, k)` thỏa:

```text
0 <= i <= j < k < n
```

ta định nghĩa:

- `s1 = s[i..j]` — đoạn từ vị trí `i` đến vị trí `j` (**tính cả hai đầu**),
- `s2 = s[j + 1..k]` — đoạn ngay sau đó, từ `j + 1` đến `k` (tính cả hai đầu).

Cặp `(s1, s2)` được gọi là **magical** (kỳ diệu) nếu **số ký tự phân biệt** (distinct characters) trong `s1` **bằng** số ký tự phân biệt trong `s2`.

Hãy trả về **số bộ ba** `(i, j, k)` mà `(s1, s2)` là cặp magical.

---

## Hình dung

```text
 vị trí:   i ......... j | j+1 ......... k
           [    s1     ] [      s2      ]
```

- `s1` và `s2` luôn **khác rỗng** (vì `i <= j` và `j + 1 <= k`).
- Hai đoạn **đứng sát nhau**: `s2` bắt đầu ngay sau chỗ `s1` kết thúc. Ta gọi `j` là **vết cắt**.
- "Số ký tự phân biệt" của `"cac"` là 2 (gồm `c` và `a`), của `"kk"` là 1, của `"abca"` là 3.
- Ta đếm **bộ ba chỉ số**, không đếm chuỗi: hai bộ ba khác nhau dù cho ra cùng nội dung `s1`, `s2` vẫn được tính 2 lần.

## Ví dụ trong đề

**Ví dụ 1:** `s = "aba"` → kết quả `2`

| STT | (i, j, k) | s1 | s2 | distinct(s1) | distinct(s2) |
|---|---|---|---|---|---|
| 1 | (0, 0, 1) | "a" | "b" | 1 | 1 |
| 2 | (1, 1, 2) | "b" | "a" | 1 | 1 |

**Ví dụ 2:** `s = "kkk"` → kết quả `4`

| STT | (i, j, k) | s1 | s2 | distinct(s1) | distinct(s2) |
|---|---|---|---|---|---|
| 1 | (0, 0, 1) | "k" | "k" | 1 | 1 |
| 2 | (0, 0, 2) | "k" | "kk" | 1 | 1 |
| 3 | (0, 1, 2) | "kk" | "k" | 1 | 1 |
| 4 | (1, 1, 2) | "k" | "k" | 1 | 1 |

**Ví dụ 3:** `s = "abcac"` → kết quả `7`

| STT | (i, j, k) | s1 | s2 | distinct(s1) | distinct(s2) |
|---|---|---|---|---|---|
| 1 | (0, 0, 1) | "a" | "b" | 1 | 1 |
| 2 | (1, 1, 2) | "b" | "c" | 1 | 1 |
| 3 | (2, 2, 3) | "c" | "a" | 1 | 1 |
| 4 | (3, 3, 4) | "a" | "c" | 1 | 1 |
| 5 | (0, 1, 3) | "ab" | "ca" | 2 | 2 |
| 6 | (0, 1, 4) | "ab" | "cac" | 2 | 2 |
| 7 | (1, 2, 4) | "bc" | "ac" | 2 | 2 |

---

## Ràng buộc (constraints)

- `2 <= s.length <= 10^5`
- `s` chỉ gồm chữ cái tiếng Anh thường.

## Ràng buộc nói gì về độ phức tạp?

**Bước 1 — có bao nhiêu bộ ba?** Với mỗi vết cắt `j` (từ `0` đến `n − 2`), có `j + 1` cách chọn `i` (từ `0` đến `j`) và `n − 1 − j` cách chọn `k` (từ `j + 1` đến `n − 1`). Tổng số bộ ba:

```text
T(n) = Σ_{j=0}^{n-2} (j + 1) · (n − 1 − j)  =  C(n + 1, 3)  ≈  n³ / 6
```

Với `n = 10^5`: `T ≈ 1.67 · 10^14`. Không máy nào duyệt nổi từng bộ ba.

**Bước 2 — ngân sách thời gian.** Một bài LeetCode thường chịu được khoảng `10^7–10^8` phép tính đơn giản; Python chậm hơn C++ nhiều lần nên ta nên nhắm cỡ vài triệu phép tính.

| Độ phức tạp | Số phép tính khi n = 10^5 | Kết luận |
|---|---|---|
| `O(n^3)` | `10^15` | không thể |
| `O(n^2)` | `10^10` | quá chậm |
| `O(n log n)` | `~1.7 · 10^6` | ổn |
| `O(26 · n)` | `2.6 · 10^6` | ổn |

**Bước 3 — dấu hiệu quan trọng nhất:** chỉ có **26 chữ cái**. Vậy số ký tự phân biệt của bất kỳ đoạn nào cũng chỉ nằm trong `1..26`. Mọi lời giải nhanh của bài này đều khai thác con số 26 nhỏ xíu đó.

**Bước 4 — đáp án rất lớn:** khi `s = "aaa...a"` thì mọi bộ ba đều magical, đáp án bằng `T(n) ≈ 1.67 · 10^14 > 2^31`. Python tự xử lý số nguyên lớn, nhưng nếu viết C++/Java thì phải dùng `long long` / `long`.

Cell dưới in dữ liệu các ví dụ và kiểm tra công thức `T(n) = C(n + 1, 3)`.

In [1]:
from math import comb

examples = [("aba", 2), ("kkk", 4), ("abcac", 7)]

def total_triplets(n: int) -> int:
    """Số bộ ba (i, j, k) với 0 <= i <= j < k < n."""
    return sum((j + 1) * (n - 1 - j) for j in range(n - 1))

for s, expected in examples:
    n = len(s)
    print(f"s = {s!r:8} n = {n} | số chữ cái khác nhau = {len(set(s))}"
          f" | tổng số bộ ba = {total_triplets(n)} | đáp án mong đợi = {expected}")

print()
for n in [10, 100, 10**5]:
    print(f"n = {n:>6}: số bộ ba = {total_triplets(n):,}  | C(n+1, 3) = {comb(n + 1, 3):,}")

s = 'aba'    n = 3 | số chữ cái khác nhau = 2 | tổng số bộ ba = 4 | đáp án mong đợi = 2
s = 'kkk'    n = 3 | số chữ cái khác nhau = 1 | tổng số bộ ba = 4 | đáp án mong đợi = 4
s = 'abcac'  n = 5 | số chữ cái khác nhau = 3 | tổng số bộ ba = 20 | đáp án mong đợi = 7

n =     10: số bộ ba = 165  | C(n+1, 3) = 165
n =    100: số bộ ba = 166,650  | C(n+1, 3) = 166,650
n = 100000: số bộ ba = 166,666,666,650,000  | C(n+1, 3) = 166,666,666,650,000


# 2. Hiểu đề bằng tay (chưa cần thuật toán)

## 2.1. Liệt kê toàn bộ với `s = "aba"`

`n = 3`, nên có `T(3) = 4` bộ ba. Ta viết hết ra:

| (i, j, k) | s1 = s[i..j] | s2 = s[j+1..k] | distinct(s1) | distinct(s2) | Magical? |
|---|---|---|---|---|---|
| (0, 0, 1) | "a" | "b" | 1 | 1 | Có |
| (0, 0, 2) | "a" | "ba" | 1 | 2 | Không |
| (0, 1, 2) | "ab" | "a" | 2 | 1 | Không |
| (1, 1, 2) | "b" | "a" | 1 | 1 | Có |

Kết quả: **2**, khớp đề.

## 2.2. Với `s = "kkk"`

Mọi đoạn con của `"kkk"` đều chỉ có 1 ký tự phân biệt, nên **cả 4 bộ ba** đều magical → **4**.

Tổng quát: nếu `s` chỉ gồm một chữ cái lặp lại, đáp án chính là `T(n) = C(n + 1, 3)`. Đây là trường hợp đáp án **lớn nhất**, ta sẽ dùng làm edge case.

Cell dưới in đủ 4 bộ ba của `"aba"` để đối chiếu với bảng.

In [2]:
def list_all_triplets(s: str) -> int:
    n = len(s)
    count = 0
    for i in range(n):
        for j in range(i, n):
            for k in range(j + 1, n):
                s1 = s[i:j + 1]
                s2 = s[j + 1:k + 1]
                d1, d2 = len(set(s1)), len(set(s2))
                ok = (d1 == d2)
                count += ok
                print(f"(i,j,k)=({i},{j},{k})  s1={s1!r:6} s2={s2!r:6} "
                      f"distinct: {d1} vs {d2}  -> {'MAGICAL' if ok else '-'}")
    print("Số bộ ba magical:", count)
    return count

count_aba = list_all_triplets("aba")

(i,j,k)=(0,0,1)  s1='a'    s2='b'    distinct: 1 vs 1  -> MAGICAL
(i,j,k)=(0,0,2)  s1='a'    s2='ba'   distinct: 1 vs 2  -> -
(i,j,k)=(0,1,2)  s1='ab'   s2='a'    distinct: 2 vs 1  -> -
(i,j,k)=(1,1,2)  s1='b'    s2='a'    distinct: 1 vs 1  -> MAGICAL
Số bộ ba magical: 2


## 2.3. Nhìn theo vết cắt `j` với `s = "abcac"`

`"abcac"` có `T(5) = 20` bộ ba — liệt kê hết thì rối. Hãy **gom theo vết cắt** `j`:

- `s1` luôn **kết thúc đúng tại `j`** → `s1` là một **hậu tố** (suffix) của phần bên trái `s[0..j]`.
- `s2` luôn **bắt đầu đúng tại `j + 1`** → `s2` là một **tiền tố** (prefix) của phần bên phải `s[j+1..n−1]`.

| j | bên trái `s[0..j]` | bên phải `s[j+1..]` | Các bộ ba magical | Số lượng |
|---|---|---|---|---|
| 0 | "a" | "bcac" | (0,0,1) | 1 |
| 1 | "ab" | "cac" | (0,1,3), (0,1,4), (1,1,2) | 3 |
| 2 | "abc" | "ac" | (1,2,4), (2,2,3) | 2 |
| 3 | "abca" | "c" | (3,3,4) | 1 |

Tổng `1 + 3 + 2 + 1 = 7`, khớp đề.

> Ghi nhớ: **chọn `j` trước**, sau đó chọn `i` ở bên trái và `k` ở bên phải. Hai lựa chọn này là **độc lập**. Đây là hạt giống của Cách 2.

In [3]:
def magical_by_cut(s: str) -> int:
    n = len(s)
    total = 0
    for j in range(n - 1):
        found = []
        for i in range(j + 1):
            for k in range(j + 1, n):
                if len(set(s[i:j + 1])) == len(set(s[j + 1:k + 1])):
                    found.append((i, j, k))
        total += len(found)
        print(f"j = {j}: trái = {s[:j + 1]!r:7} phải = {s[j + 1:]!r:7} -> {len(found)} bộ: {found}")
    print("Tổng:", total)
    return total

total_abcac = magical_by_cut("abcac")

j = 0: trái = 'a'     phải = 'bcac'  -> 1 bộ: [(0, 0, 1)]
j = 1: trái = 'ab'    phải = 'cac'   -> 3 bộ: [(0, 1, 3), (0, 1, 4), (1, 1, 2)]
j = 2: trái = 'abc'   phải = 'ac'    -> 2 bộ: [(1, 2, 4), (2, 2, 3)]
j = 3: trái = 'abca'  phải = 'c'     -> 1 bộ: [(3, 3, 4)]
Tổng: 7


# 3. Nền tảng: các công cụ Python và ý tưởng sẽ dùng

Trước khi vào lời giải, ta học từng "viên gạch". Mỗi mục: **nó là gì → ví dụ nhỏ → code chạy thử → liên quan gì tới bài**.

| Mục | Công cụ / ý tưởng | Dùng ở đâu |
|---|---|---|
| 3.1 | Ký hiệu `s[i..j]` và slicing `s[i:j+1]` | mọi cách |
| 3.2 | `set` và `len(set(...))` để đếm ký tự phân biệt | Cách 1, 2 |
| 3.3 | Mảng đếm 26 ô, `ord(c) - ord('a')` | Cách 4, Solution |
| 3.4 | Nguyên lý cộng và nguyên lý nhân | Cách 2, 3, 4 |
| 3.5 | Tính đơn điệu của số ký tự phân biệt | Cách 3, 4 |
| 3.6 | `dict` lưu vị trí cuối cùng, `sorted`, lính canh, `zip`, `bisect.insort` | Cách 3, bài tập |
| 3.7 | Cửa sổ trượt / hai con trỏ (sliding window) | Cách 4 |
| 3.8 | Đảo chuỗi `[::-1]` và ánh xạ chỉ số | Cách 4 |

## 3.1. `s[i..j]` (đóng hai đầu) và slicing `s[i:j+1]`

Đề bài viết `s[i..j]` nghĩa là **lấy cả vị trí `i` lẫn vị trí `j`**. Trong Python, slicing `s[a:b]` lấy từ `a` đến **trước** `b` (không lấy `b`).

Vì vậy:

```text
s[i..j]   (đề bài)    ==   s[i:j + 1]   (Python)
độ dài = j − i + 1
```

Ví dụ `s = "abcac"`:

- `s[1..3]` = `"bca"` = `s[1:4]` (độ dài 3).
- Nếu lỡ viết `s[1:3]` thì chỉ được `"bc"` — thiếu ký tự cuối. Đây là lỗi off-by-one số 1 của bài.

In [4]:
s = "abcac"
i, j = 1, 3
print("s[i:j+1] =", repr(s[i:j + 1]), "| độ dài =", len(s[i:j + 1]), "| j - i + 1 =", j - i + 1)
print("s[i:j]   =", repr(s[i:j]), " <- thiếu s[j], SAI")

# s2 = s[j+1..k] với k = 4
k = 4
print("s2 = s[j+1:k+1] =", repr(s[j + 1:k + 1]))

s[i:j+1] = 'bca' | độ dài = 3 | j - i + 1 = 3
s[i:j]   = 'bc'  <- thiếu s[j], SAI
s2 = s[j+1:k+1] = 'c'


## 3.2. `set` và đếm ký tự phân biệt

`set` (tập hợp) chỉ giữ **mỗi phần tử một lần**. Vì vậy:

```python
len(set("cac"))   # {'c', 'a'} -> 2
```

Có hai cách dùng:

1. **Tính lại từ đầu**: `len(set(s[i:j+1]))` — tốn `O(độ dài)` mỗi lần (Cách 1).
2. **Tăng dần**: khi kéo dài một đoạn thêm 1 ký tự, chỉ cần `seen.add(ch)` rồi đọc `len(seen)` — mỗi bước `O(1)` trung bình (Cách 2).

Lưu ý: `set` chỉ hợp khi đoạn **chỉ dài ra**. Nếu đoạn vừa dài ra vừa bị cắt bớt ở đầu kia (cửa sổ trượt), `set` không biết ký tự bị bỏ có còn bản sao nào trong cửa sổ không — lúc đó cần **mảng đếm** (mục 3.3).

In [5]:
print(len(set("cac")), len(set("kk")), len(set("abca")))

# Kéo dài dần đoạn s[0..r] và theo dõi số ký tự phân biệt
s = "abcac"
seen = set()
for r, ch in enumerate(s):
    seen.add(ch)
    print(f"r = {r}, thêm {ch!r}: đoạn = {s[:r + 1]!r:8} distinct = {len(seen)}")

2 1 3
r = 0, thêm 'a': đoạn = 'a'      distinct = 1
r = 1, thêm 'b': đoạn = 'ab'     distinct = 2
r = 2, thêm 'c': đoạn = 'abc'    distinct = 3
r = 3, thêm 'a': đoạn = 'abca'   distinct = 3
r = 4, thêm 'c': đoạn = 'abcac'  distinct = 3


## 3.3. Mảng đếm 26 ô và `ord`

`ord(ch)` trả về mã số của ký tự: `ord('a') = 97`, `ord('b') = 98`, … Vậy `ord(ch) - ord('a')` đổi `'a'..'z'` thành `0..25` — dùng làm chỉ số cho một list 26 ô.

```python
cnt = [0] * 26          # cnt[c] = số lần chữ c xuất hiện trong cửa sổ
```

Theo dõi số ký tự phân biệt `distinct` khi **thêm** và **bớt**:

- Thêm chữ `c`: nếu `cnt[c]` đang là `0` → đây là chữ mới → `distinct += 1`. Sau đó `cnt[c] += 1`.
- Bớt chữ `c`: `cnt[c] -= 1`; nếu `cnt[c]` về `0` → chữ đó biến mất → `distinct -= 1`.

Đây chính là "bộ đếm" bên trong cửa sổ trượt của Cách 4 và trong bản Solution.

In [6]:
print(ord('a'), ord('c'), ord('c') - ord('a'))

cnt = [0] * 26
distinct = 0

def add(ch):
    global distinct
    c = ord(ch) - ord('a')
    if cnt[c] == 0:
        distinct += 1
    cnt[c] += 1

def remove(ch):
    global distinct
    c = ord(ch) - ord('a')
    cnt[c] -= 1
    if cnt[c] == 0:
        distinct -= 1

for ch in "cac":
    add(ch)
print("Sau khi thêm 'c','a','c': distinct =", distinct, "| cnt[c] =", cnt[2], "| cnt[a] =", cnt[0])
remove('c')
print("Bớt một 'c'        : distinct =", distinct, "(vẫn còn một 'c')")
remove('c')
print("Bớt 'c' thứ hai    : distinct =", distinct, "(chỉ còn 'a')")

97 99 2
Sau khi thêm 'c','a','c': distinct = 2 | cnt[c] = 2 | cnt[a] = 1
Bớt một 'c'        : distinct = 2 (vẫn còn một 'c')
Bớt 'c' thứ hai    : distinct = 1 (chỉ còn 'a')


## 3.4. Nguyên lý cộng và nguyên lý nhân

**Nguyên lý nhân** (multiplication principle): nếu việc A có `a` cách, việc B có `b` cách, và hai việc **độc lập** với nhau, thì có `a · b` cách làm cả hai.

> Ví dụ: 2 áo × 3 quần = 6 bộ đồ.

**Nguyên lý cộng** (addition principle): nếu chia các trường hợp thành nhiều nhóm **không giao nhau**, tổng số = tổng của từng nhóm.

Áp dụng vào bài, với một vết cắt `j` cố định:

- Gọi `L[d]` = số cách chọn `i` sao cho `s[i..j]` có **đúng** `d` ký tự phân biệt.
- Gọi `R[d]` = số cách chọn `k` sao cho `s[j+1..k]` có **đúng** `d` ký tự phân biệt.
- Cặp magical "cùng mức `d`" có `L[d] · R[d]` cách (nguyên lý nhân).
- Các mức `d = 1, 2, …, 26` không giao nhau → cộng lại (nguyên lý cộng):

```text
đóng góp của vết cắt j = Σ_d  L[d] · R[d]
```

Ví dụ `s = "abcac"`, `j = 1`: bên trái `"ab"` có hậu tố `"b"` (1), `"ab"` (2) → `L = [1, 1]`. Bên phải `"cac"` có tiền tố `"c"` (1), `"ca"` (2), `"cac"` (2) → `R = [1, 2]`. Đóng góp = `1·1 + 1·2 = 3` — khớp bảng mục 2.3.

In [7]:
from itertools import product

left_suffixes = ["b", "ab"]          # các s1 có thể khi j = 1
right_prefixes = ["c", "ca", "cac"]  # các s2 có thể khi j = 1

# Cách 1: thử mọi cặp (nguyên lý nhân "thô")
pairs = [(a, b) for a, b in product(left_suffixes, right_prefixes) if len(set(a)) == len(set(b))]
print("Các cặp magical:", pairs, "->", len(pairs))

# Cách 2: đếm theo mức d rồi nhân
L = [0] * 27
R = [0] * 27
for a in left_suffixes:
    L[len(set(a))] += 1
for b in right_prefixes:
    R[len(set(b))] += 1
print("L[1..3] =", L[1:4], "| R[1..3] =", R[1:4])
print("Σ L[d]·R[d] =", sum(L[d] * R[d] for d in range(1, 27)))

Các cặp magical: [('b', 'c'), ('ab', 'ca'), ('ab', 'cac')] -> 3
L[1..3] = [1, 1, 0] | R[1..3] = [1, 2, 0]
Σ L[d]·R[d] = 3


## 3.5. Tính đơn điệu (monotonic)

**Quan sát:** nếu đoạn `A` nằm trong đoạn `B` thì `distinct(A) <= distinct(B)`. Thêm ký tự không bao giờ làm **mất** chữ cái nào.

Hệ quả: cố định điểm cuối `j`, cho `i` chạy **lùi dần** từ `j` về `0` (đoạn `s[i..j]` dài ra), dãy `distinct(s[i..j])`:

- bắt đầu bằng `1` (đoạn chỉ có `s[j]`),
- **không bao giờ giảm**,
- mỗi bước tăng **tối đa 1** (chỉ thêm đúng một ký tự),
- không vượt quá `26`.

Ví dụ `s = "abcac"`, `j = 4`: các hậu tố `"c", "ac", "cac", "bcac", "abcac"` có distinct lần lượt `1, 2, 2, 3, 3`.

Một dãy như vậy trông như **bậc thang** đi lên, có tối đa 26 bậc. Đây là chìa khóa của Cách 3 và Cách 4.

In [8]:
s = "abcac"
j = 4
for i in range(j, -1, -1):
    print(f"i = {i}: s[i..j] = {s[i:j + 1]!r:8} distinct = {len(set(s[i:j + 1]))}")

seq = [len(set(s[i:j + 1])) for i in range(j, -1, -1)]
print("Dãy distinct khi i lùi dần:", seq)
print("Không giảm?", all(seq[t] <= seq[t + 1] for t in range(len(seq) - 1)))
print("Mỗi bước tăng tối đa 1?", all(seq[t + 1] - seq[t] <= 1 for t in range(len(seq) - 1)))

i = 4: s[i..j] = 'c'      distinct = 1
i = 3: s[i..j] = 'ac'     distinct = 2
i = 2: s[i..j] = 'cac'    distinct = 2
i = 1: s[i..j] = 'bcac'   distinct = 3
i = 0: s[i..j] = 'abcac'  distinct = 3
Dãy distinct khi i lùi dần: [1, 2, 2, 3, 3]
Không giảm? True
Mỗi bước tăng tối đa 1? True


## 3.6. `dict` lưu vị trí cuối cùng, `sorted`, lính canh, `zip`, `bisect.insort`

**(a) Vị trí xuất hiện cuối cùng (last occurrence).** Duyệt từ trái sang phải và **ghi đè**:

```python
last = {}
for i, ch in enumerate(s):
    last[ch] = i        # lần sau gặp lại ch sẽ ghi đè vị trí mới hơn
```

Sau vòng lặp, `last[ch]` là vị trí **cuối cùng** của `ch`. Duyệt từ phải sang trái và ghi đè thì ngược lại: được vị trí **đầu tiên**.

**(b) `sorted(values, reverse=True)`** sắp xếp giảm dần. Chú ý: thứ tự trong `dict` là **thứ tự chèn** (insertion order), **không phải** thứ tự vị trí — nên phải sắp xếp.

**(c) Lính canh (sentinel):** thêm một giá trị giả ở cuối danh sách (ví dụ `-1` hoặc `n`) để công thức "hiệu hai phần tử liên tiếp" chạy đều cho cả phần tử cuối, khỏi viết `if` riêng.

**(d) `zip(A, B)`** ghép từng cặp phần tử và **dừng ở danh sách ngắn hơn** — rất tiện khi một bên có ít mức `d` hơn bên kia.

**(e) `bisect.insort(lst, x)`** chèn `x` vào list đã sắp xếp mà vẫn giữ thứ tự (dùng ở bài tập nâng cao).

In [9]:
import bisect

s = "abca"
last = {}
for i, ch in enumerate(s):
    last[ch] = i
print("last =", last, "(thứ tự chèn, chưa sắp xếp)")

p = sorted(last.values(), reverse=True)
p.append(-1)                      # lính canh
print("p (giảm dần + lính canh -1) =", p)
diffs = [p[t] - p[t + 1] for t in range(len(p) - 1)]
print("hiệu liên tiếp =", diffs, "| tổng =", sum(diffs), "= len(s)")

A, B = [1, 1, 2], [1, 5]
print("zip dừng ở list ngắn hơn:", list(zip(A, B)), "-> Σ a*b =", sum(a * b for a, b in zip(A, B)))

q = [2, 5, 9]
bisect.insort(q, 6)
print("insort 6 vào [2, 5, 9] ->", q)

last = {'a': 3, 'b': 1, 'c': 2} (thứ tự chèn, chưa sắp xếp)
p (giảm dần + lính canh -1) = [3, 2, 1, -1]
hiệu liên tiếp = [1, 1, 2] | tổng = 4 = len(s)
zip dừng ở list ngắn hơn: [(1, 1), (1, 5)] -> Σ a*b = 6
insort 6 vào [2, 5, 9] -> [2, 5, 6, 9]


## 3.7. Cửa sổ trượt / hai con trỏ (sliding window / two pointers)

Bài toán con kinh điển: *"Với mỗi vị trí `r`, tìm đoạn **dài nhất kết thúc tại `r`** có **nhiều nhất `d`** ký tự phân biệt."*

Ý tưởng hai con trỏ `l` (trái) và `r` (phải):

1. Tăng `r`, thêm `s[r]` vào cửa sổ (cập nhật mảng đếm).
2. **Trong khi** cửa sổ có quá `d` ký tự phân biệt: bỏ `s[l]` ra, tăng `l`.
3. Lúc này `s[l..r]` là đoạn dài nhất kết thúc tại `r` thỏa điều kiện; độ dài `r − l + 1`.

**Vì sao nhanh?** `l` **chỉ tiến, không bao giờ lùi** (mục 3.5: nếu `s[l..r]` đã quá `d` chữ thì `s[l..r+1]` càng quá). Tổng số lần `l` tiến ≤ `n`, nên cả vòng lặp là `O(n)` dù có `while` lồng bên trong.

Ví dụ quen thuộc (LeetCode 159): đoạn dài nhất có nhiều nhất 2 ký tự phân biệt trong `"eceba"` là `"ece"`, độ dài 3.

In [10]:
def longest_at_most_two(s: str) -> int:
    cnt = {}
    l = 0
    best = 0
    for r, ch in enumerate(s):
        cnt[ch] = cnt.get(ch, 0) + 1
        while len(cnt) > 2:              # quá 2 chữ phân biệt -> co cửa sổ
            cnt[s[l]] -= 1
            if cnt[s[l]] == 0:
                del cnt[s[l]]
            l += 1
        best = max(best, r - l + 1)
        print(f"r = {r} ({ch!r}): cửa sổ = s[{l}..{r}] = {s[l:r + 1]!r:6} độ dài = {r - l + 1}")
    return best

print("Dài nhất:", longest_at_most_two("eceba"))

r = 0 ('e'): cửa sổ = s[0..0] = 'e'    độ dài = 1
r = 1 ('c'): cửa sổ = s[0..1] = 'ec'   độ dài = 2
r = 2 ('e'): cửa sổ = s[0..2] = 'ece'  độ dài = 3
r = 3 ('b'): cửa sổ = s[2..3] = 'eb'   độ dài = 2
r = 4 ('a'): cửa sổ = s[3..4] = 'ba'   độ dài = 2
Dài nhất: 3


## 3.8. Đảo chuỗi `[::-1]` và ánh xạ chỉ số

`s[::-1]` tạo chuỗi đảo ngược. Vị trí `x` trong `s` tương ứng vị trí `n − 1 − x` trong `rev = s[::-1]`.

**Vì sao cần?** Bên phải vết cắt ta cần đếm các **tiền tố** của `s[j+1..]` (đoạn bắt đầu tại `j + 1`). Đảo chuỗi lại thì "tiền tố bắt đầu tại `j + 1`" biến thành "**hậu tố kết thúc tại** `n − 1 − (j + 1) = n − 2 − j`" của `rev`. Thế là bên phải cũng dùng lại được đúng code của bên trái!

Ví dụ `s = "abcac"` (`n = 5`), `j = 1`: các tiền tố của `s[2..4] = "cac"` là `"c", "ca", "cac"`. Trong `rev = "cacba"`, các hậu tố kết thúc tại `n − 2 − j = 2` là `"c", "ac", "cac"` — chính là các chuỗi đó viết ngược, nên **số ký tự phân biệt giống hệt**.

In [11]:
s = "abcac"
n = len(s)
rev = s[::-1]
j = 1
r = n - 2 - j
print("rev =", rev, "| vị trí tương ứng của j+1 trong rev =", r)
for k in range(j + 1, n):
    prefix = s[j + 1:k + 1]
    start_in_rev = n - 1 - k
    suffix_rev = rev[start_in_rev:r + 1]
    print(f"tiền tố {prefix!r:6} <-> hậu tố của rev {suffix_rev!r:6} | distinct: {len(set(prefix))} = {len(set(suffix_rev))}")

rev = cacba | vị trí tương ứng của j+1 trong rev = 2
tiền tố 'c'    <-> hậu tố của rev 'c'    | distinct: 1 = 1
tiền tố 'ca'   <-> hậu tố của rev 'ac'   | distinct: 2 = 2
tiền tố 'cac'  <-> hậu tố của rev 'cac'  | distinct: 2 = 2


# 4. Cách 1 — Brute Force: thử mọi bộ ba

## Ý tưởng

Làm đúng như đề nói, không suy nghĩ gì thêm:

1. Duyệt `i` từ `0` đến `n − 1`.
2. Duyệt `j` từ `i` đến `n − 1` (vì `i <= j`).
3. Duyệt `k` từ `j + 1` đến `n − 1` (vì `j < k`).
4. Cắt `s1 = s[i:j+1]`, `s2 = s[j+1:k+1]`, so sánh `len(set(s1)) == len(set(s2))`.

Cách này **hiển nhiên đúng** vì nó dịch đề bài sang code từng chữ. Ta sẽ dùng nó làm **trọng tài** (referee) để kiểm tra các cách nhanh hơn.

In [12]:
def count_magical_bruteforce(s: str) -> int:
    n = len(s)
    ans = 0
    for i in range(n):
        for j in range(i, n):
            for k in range(j + 1, n):
                s1 = s[i:j + 1]
                s2 = s[j + 1:k + 1]
                if len(set(s1)) == len(set(s2)):
                    ans += 1
    return ans

for s, expected in examples:
    got = count_magical_bruteforce(s)
    print(f"{s!r:8} -> {got} (mong đợi {expected})")
    assert got == expected
print("Cách 1 đúng trên cả 3 ví dụ.")

'aba'    -> 2 (mong đợi 2)
'kkk'    -> 4 (mong đợi 4)
'abcac'  -> 7 (mong đợi 7)
Cách 1 đúng trên cả 3 ví dụ.


## Cách 1 tốn bao nhiêu công?

- Số bộ ba: `T(n) ≈ n³ / 6`.
- Mỗi bộ ba, ta đưa `(j − i + 1) + (k − j) = k − i + 1` ký tự vào `set`.

Gom theo độ dài `len = k − i + 1` của đoạn `s[i..k]`: có `n − len + 1` cặp `(i, k)` và `len − 1` vị trí cắt `j` bên trong, mỗi lần tốn `len` ký tự. Tổng ≈ `n⁴ / 12`.

Vậy:

```text
Time  : O(n^4)
Space : O(n)   (chuỗi con tạm thời)
```

Cell dưới tính chính xác tổng số lượt thêm ký tự vào `set`.

In [13]:
def brute_force_work(n: int) -> int:
    """Tổng số ký tự mà Cách 1 phải đưa vào set."""
    total = 0
    for length in range(2, n + 1):       # length = k - i + 1
        ways_ik = n - length + 1          # số cặp (i, k) có độ dài này
        ways_j = length - 1               # số vị trí cắt j bên trong
        total += ways_ik * ways_j * length
    return total

for n in [10, 100, 1000, 10**5]:
    print(f"n = {n:>6}: {brute_force_work(n):.2e} lượt thêm ký tự | n^4/12 = {n**4 / 12:.2e}")

n =     10: 9.90e+02 lượt thêm ký tự | n^4/12 = 8.33e+02
n =    100: 8.50e+06 lượt thêm ký tự | n^4/12 = 8.33e+06
n =   1000: 8.35e+10 lượt thêm ký tự | n^4/12 = 8.33e+10
n = 100000: 8.33e+18 lượt thêm ký tự | n^4/12 = 8.33e+18


Với `n = 10^5`, con số là cỡ `8 · 10^18` — hoàn toàn bất khả thi. Ngay cả `n = 1000` đã cần `~8 · 10^10` lượt.

**Nút thắt của Cách 1** có hai tầng:

1. Với mỗi bộ ba, ta **tính lại** số ký tự phân biệt từ đầu — trong khi các đoạn lồng nhau chia sẻ gần hết ký tự.
2. Ta **thử từng cặp** `(i, k)` một — trong khi điều kiện chỉ phụ thuộc vào hai **con số** `distinct(s1)` và `distinct(s2)`.

Bước đột phá đầu tiên xử lý tầng thứ 2.

# 5. Bước đột phá 1 — cố định vết cắt `j`, hai bên độc lập

## Suy luận từng bước

**Bước 1.** Mỗi bộ ba `(i, j, k)` được xác định bởi: chọn vết cắt `j` (từ `0` đến `n − 2`), rồi chọn `i` trong `[0, j]`, rồi chọn `k` trong `[j + 1, n − 1]`.

**Bước 2.** Khi `j` đã cố định, `s1 = s[i..j]` chỉ phụ thuộc `i`; `s2 = s[j+1..k]` chỉ phụ thuộc `k`. Hai lựa chọn **không ảnh hưởng nhau**.

**Bước 3.** Điều kiện magical chỉ hỏi `distinct(s1) == distinct(s2)`. Ta không cần biết `i`, `k` cụ thể là gì, chỉ cần biết **bao nhiêu** `i` cho ra mỗi giá trị `d`, và **bao nhiêu** `k` cho ra mỗi giá trị `d`.

**Bước 4.** Định nghĩa, với vết cắt `j`:

```text
L_j[d] = số i ∈ [0, j]       sao cho s[i..j]   có đúng d ký tự phân biệt
R_j[d] = số k ∈ [j+1, n−1]   sao cho s[j+1..k] có đúng d ký tự phân biệt
```

**Bước 5.** Theo nguyên lý nhân và nguyên lý cộng (mục 3.4):

```text
Đáp án = Σ_{j=0}^{n-2}  Σ_{d=1}^{26}  L_j[d] · R_j[d]
```

## Kiểm chứng trên `s = "abcac"`

| j | trái `s[0..j]` | `L_j` (d = 1, 2, 3) | phải `s[j+1..]` | `R_j` (d = 1, 2, 3) | Σ L·R |
|---|---|---|---|---|---|
| 0 | "a" | [1, 0, 0] | "bcac" | [1, 1, 2] | 1·1 = 1 |
| 1 | "ab" | [1, 1, 0] | "cac" | [1, 2, 0] | 1·1 + 1·2 = 3 |
| 2 | "abc" | [1, 1, 1] | "ac" | [1, 1, 0] | 1·1 + 1·1 = 2 |
| 3 | "abca" | [1, 1, 2] | "c" | [1, 0, 0] | 1·1 = 1 |

Tổng = `1 + 3 + 2 + 1 = 7`. Cell dưới tính bảng này bằng code.

In [14]:
def side_counts(s: str, j: int):
    """Trả về (L_j, R_j) dạng list 27 ô, ô d = số đoạn có đúng d ký tự phân biệt."""
    n = len(s)
    L = [0] * 27
    for i in range(j + 1):
        L[len(set(s[i:j + 1]))] += 1
    R = [0] * 27
    for k in range(j + 1, n):
        R[len(set(s[j + 1:k + 1]))] += 1
    return L, R

s = "abcac"
total = 0
for j in range(len(s) - 1):
    L, R = side_counts(s, j)
    contrib = sum(L[d] * R[d] for d in range(1, 27))
    total += contrib
    print(f"j = {j}: L[1..3] = {L[1:4]}  R[1..3] = {R[1:4]}  -> Σ L·R = {contrib}")
print("Tổng =", total)

j = 0: L[1..3] = [1, 0, 0]  R[1..3] = [1, 1, 2]  -> Σ L·R = 1
j = 1: L[1..3] = [1, 1, 0]  R[1..3] = [1, 2, 0]  -> Σ L·R = 3
j = 2: L[1..3] = [1, 1, 1]  R[1..3] = [1, 1, 0]  -> Σ L·R = 2
j = 3: L[1..3] = [1, 1, 2]  R[1..3] = [1, 0, 0]  -> Σ L·R = 1
Tổng = 7


# 6. Cách 2 — Đếm theo vết cắt: `O(n²)`

## Ý tưởng

Với mỗi vết cắt `j`:

1. **Bên trái:** cho `i` chạy lùi từ `j` về `0`, `seen.add(s[i])` (mục 3.2). Sau mỗi bước, `len(seen)` chính là `distinct(s[i..j])` → `cnt_left[len(seen)] += 1`.
2. **Bên phải:** cho `k` chạy từ `j + 1` đến `n − 1`, `seen.add(s[k])` → `cnt_right[len(seen)] += 1`.
3. Cộng `Σ_d cnt_left[d] · cnt_right[d]` vào đáp án.

So với Cách 1, ta đã bỏ được:

- việc tính lại `set` từ đầu (nhờ **kéo dài dần**),
- việc thử từng cặp `(i, k)` (nhờ **đếm theo mức `d` rồi nhân**).

In [15]:
def count_magical_split(s: str) -> int:
    n = len(s)
    ans = 0
    for j in range(n - 1):
        # Bên trái: các hậu tố s[i..j], i chạy lùi
        cnt_left = [0] * 27
        seen = set()
        for i in range(j, -1, -1):
            seen.add(s[i])
            cnt_left[len(seen)] += 1
        # Bên phải: các tiền tố s[j+1..k], k chạy tới
        cnt_right = [0] * 27
        seen = set()
        for k in range(j + 1, n):
            seen.add(s[k])
            cnt_right[len(seen)] += 1
        # Nguyên lý nhân + cộng
        ans += sum(cnt_left[d] * cnt_right[d] for d in range(1, 27))
    return ans

for s, expected in examples:
    got = count_magical_split(s)
    print(f"{s!r:8} -> {got} (mong đợi {expected})")
    assert got == expected

'aba'    -> 2 (mong đợi 2)
'kkk'    -> 4 (mong đợi 4)
'abcac'  -> 7 (mong đợi 7)


## Độ phức tạp Cách 2

Với mỗi `j`, hai vòng lặp trong đi tổng cộng `(j + 1) + (n − 1 − j) = n` bước. Có `n − 1` vết cắt.

```text
Time  : O(n^2)      (đúng n·(n − 1) lượt seen.add)
Space : O(1)        (hai mảng 27 ô và một set ≤ 26 phần tử)
```

Với `n = 10^5` là khoảng `10^10` lượt — vẫn quá chậm: Python chỉ làm cỡ `10^7` thao tác kiểu `seen.add` mỗi giây, nên sẽ mất cỡ `10^3` giây (khoảng mười mấy phút), trong khi giới hạn chỉ vài giây.

**Lãng phí còn lại ở đâu?** Với mỗi `j`, ta duyệt lại **toàn bộ** bên trái chỉ để tạo ra mảng `cnt_left` — một mảng mà theo mục 3.5 chỉ mô tả một **bậc thang có tối đa 26 bậc**. Liệu có cách nào đọc thẳng được 26 "điểm gãy" đó mà không duyệt hết?

In [16]:
for n in [10, 1000, 10**5]:
    work = n * (n - 1)
    print(f"n = {n:>6}: Cách 2 cần {work:,} lượt seen.add")

n =     10: Cách 2 cần 90 lượt seen.add
n =   1000: Cách 2 cần 999,000 lượt seen.add
n = 100000: Cách 2 cần 9,999,900,000 lượt seen.add


# 7. Bước đột phá 2 — bậc thang chỉ gãy tại vị trí xuất hiện cuối cùng

## Khi nào bậc thang "nhảy"?

Cố định điểm cuối `j`, cho `i` lùi dần. Khi thêm `s[i]` vào đầu đoạn:

- Nếu chữ `s[i]` **đã có** trong `s[i+1..j]` → số ký tự phân biệt **giữ nguyên**.
- Nếu chữ `s[i]` **chưa có** trong `s[i+1..j]` → số ký tự phân biệt **tăng 1**.

"`s[i]` chưa có trong `s[i+1..j]`" nghĩa là: `i` là vị trí **xuất hiện cuối cùng** (last occurrence) của chữ `s[i]` trong đoạn `s[0..j]`.

> Kết luận: bậc thang nhảy **đúng tại** các vị trí xuất hiện cuối cùng của từng chữ cái. Mỗi chữ cái có đúng một vị trí như vậy → tối đa **26** điểm gãy.

## Ví dụ `s = "dabcbbac"`, `j = 7`

| i | 7 | 6 | 5 | 4 | 3 | 2 | 1 | 0 |
|---|---|---|---|---|---|---|---|---|
| s[i] | c | a | b | b | c | b | a | d |
| distinct(s[i..7]) | **1** | **2** | **3** | 3 | 3 | 3 | 3 | **4** |

Các lần "lên bậc" (in đậm, tính cả bậc đầu tiên từ 0 lên 1) xảy ra tại `i = 7, 6, 5, 0`. Vị trí cuối cùng của từng chữ: `last = {d: 0, a: 6, b: 5, c: 7}`. Sắp xếp giảm dần:

```text
p = [7, 6, 5, 0]     rồi thêm lính canh  -> [7, 6, 5, 0, -1]
     p1 p2 p3 p4
```

Số `i` cho ra **đúng** `d` ký tự phân biệt là các `i` thuộc khoảng `(p_{d+1}, p_d]`, tức có `p_d − p_{d+1}` giá trị:

| d | Khoảng i | Số lượng `p_d − p_{d+1}` |
|---|---|---|
| 1 | i = 7 | 7 − 6 = 1 |
| 2 | i = 6 | 6 − 5 = 1 |
| 3 | i = 5, 4, 3, 2, 1 | 5 − 0 = 5 |
| 4 | i = 0 | 0 − (−1) = 1 |

Tổng `1 + 1 + 5 + 1 = 8 = j + 1` (đủ mọi `i`). Lính canh `−1` xuất hiện vì `i` được phép xuống tới `0`, nên bậc cuối gồm `p_m, p_m − 1, …, 0` — tức `p_m − (−1)` giá trị.

In [17]:
s = "dabcbbac"
j = 7
staircase = [len(set(s[i:j + 1])) for i in range(j, -1, -1)]
print("distinct(s[i..j]) với i = 7, 6, ..., 0:", staircase)

last = {}
for i in range(j + 1):
    last[s[i]] = i
p = sorted(last.values(), reverse=True) + [-1]
print("last =", last)
print("p (giảm dần + lính canh) =", p)

from_breakpoints = [p[d] - p[d + 1] for d in range(len(p) - 1)]
direct = [staircase.count(d) for d in range(1, max(staircase) + 1)]
print("Đếm bằng điểm gãy :", from_breakpoints)
print("Đếm trực tiếp     :", direct)
assert from_breakpoints == direct

distinct(s[i..j]) với i = 7, 6, ..., 0: [1, 2, 3, 3, 3, 3, 3, 4]
last = {'d': 0, 'a': 6, 'b': 5, 'c': 7}
p (giảm dần + lính canh) = [7, 6, 5, 0, -1]
Đếm bằng điểm gãy : [1, 1, 5, 1]
Đếm trực tiếp     : [1, 1, 5, 1]


## Bên phải: gương soi của bên trái

Bên phải ta xét các đoạn **bắt đầu** tại `t = j + 1` và kéo dài sang phải. Lập luận y hệt, đổi chiều:

- Số ký tự phân biệt của `s[t..k]` tăng 1 khi `k` là vị trí **xuất hiện đầu tiên** (first occurrence) của chữ `s[k]` trong `s[t..n−1]`.
- Sắp xếp các vị trí đầu tiên **tăng dần**: `q_1 = t < q_2 < … < q_m`, thêm lính canh `q_{m+1} = n`.
- Số `k` cho ra đúng `d` ký tự phân biệt là `q_{d+1} − q_d`.

Ví dụ `s = "abcac"`, `t = 2` (tức `j = 1`): `s[2..4] = "cac"`, vị trí đầu tiên `{c: 2, a: 3}` → `q = [2, 3]` + lính canh `5` → `R = [3 − 2, 5 − 3] = [1, 2]`. Khớp `R_1` ở bảng mục 5.

Cell dưới kiểm tra công thức bên phải cho mọi `t` của `"abcac"`.

In [18]:
s = "abcac"
n = len(s)
for t in range(1, n):
    first = {}
    for k in range(n - 1, t - 1, -1):     # duyệt ngược + ghi đè -> vị trí đầu tiên
        first[s[k]] = k
    q = sorted(first.values()) + [n]
    from_breakpoints = [q[d + 1] - q[d] for d in range(len(q) - 1)]
    seq = [len(set(s[t:k + 1])) for k in range(t, n)]
    direct = [seq.count(d) for d in range(1, max(seq) + 1)]
    print(f"t = {t}: s[t..] = {s[t:]!r:7} q = {q}  R = {from_breakpoints}  (trực tiếp: {direct})")
    assert from_breakpoints == direct

t = 1: s[t..] = 'bcac'  q = [1, 2, 3, 5]  R = [1, 1, 2]  (trực tiếp: [1, 1, 2])
t = 2: s[t..] = 'cac'   q = [2, 3, 5]  R = [1, 2]  (trực tiếp: [1, 2])
t = 3: s[t..] = 'ac'    q = [3, 4, 5]  R = [1, 1]  (trực tiếp: [1, 1])
t = 4: s[t..] = 'c'     q = [4, 5]  R = [1]  (trực tiếp: [1])


# 8. Cách 3 — Điểm gãy (breakpoints): `O(26 · n)`

## Thuật toán

**Lượt 1 — từ phải sang trái, chuẩn bị bên phải.** Cho `t` chạy từ `n − 1` về `0`, ghi đè `first[s[t]] = t`. Vì `t` giảm dần, sau khi ghi đè thì `first[c]` luôn là vị trí **đầu tiên** của `c` trong `s[t..]`. Lưu lại:

```text
right_pos[t] = sorted(first.values()) + [n]
```

**Lượt 2 — từ trái sang phải, ghép hai phía.** Cho `j` chạy từ `0` đến `n − 2`, ghi đè `last[s[j]] = j`. Vì `j` tăng dần, `last[c]` luôn là vị trí **cuối cùng** của `c` trong `s[0..j]`. Khi đó:

```text
p = sorted(last.values(), reverse=True) + [-1]
q = right_pos[j + 1]
L[d] = p[d] - p[d+1]          (d đánh số từ 0: L[0] ứng với "1 ký tự phân biệt")
R[d] = q[d+1] - q[d]
ans += Σ L[d] · R[d]           (zip tự dừng ở bên có ít mức hơn)
```

Vì sao chỉ cần nhân tới bên ngắn hơn? Nếu bên trái chỉ có tối đa `m` ký tự phân biệt thì `L[d] = 0` với mọi `d > m` — các tích đó bằng 0, bỏ qua được.

## Lưu ý về chỉ số

- `j` chỉ chạy tới `n − 2` vì phải còn ít nhất một ký tự cho `s2` (`k >= j + 1`). Nhờ vậy `right_pos[j + 1]` luôn hợp lệ.
- Các list `p`, `q` dài tối đa 27 phần tử, nên mỗi bước chỉ tốn một lượng công việc **hằng số** (không phụ thuộc `n`).

In [19]:
def count_magical_breakpoints(s: str) -> int:
    n = len(s)

    # Lượt 1: right_pos[t] = các vị trí xuất hiện đầu tiên trong s[t..] (tăng dần) + lính canh n
    right_pos = [None] * n
    first = {}
    for t in range(n - 1, -1, -1):
        first[s[t]] = t
        q = sorted(first.values())
        q.append(n)
        right_pos[t] = q

    # Lượt 2: duyệt vết cắt j, ghép bên trái (last occurrence) với bên phải
    last = {}
    ans = 0
    for j in range(n - 1):
        last[s[j]] = j
        p = sorted(last.values(), reverse=True)
        p.append(-1)
        q = right_pos[j + 1]
        L = [p[d] - p[d + 1] for d in range(len(p) - 1)]
        R = [q[d + 1] - q[d] for d in range(len(q) - 1)]
        ans += sum(a * b for a, b in zip(L, R))
    return ans

for s, expected in examples:
    got = count_magical_breakpoints(s)
    print(f"{s!r:8} -> {got} (mong đợi {expected})")
    assert got == expected

'aba'    -> 2 (mong đợi 2)
'kkk'    -> 4 (mong đợi 4)
'abcac'  -> 7 (mong đợi 7)


## Độ phức tạp Cách 3

Gọi `n = len(s)`, `σ = 26` (kích thước bảng chữ cái).

- Mỗi bước của mỗi lượt: sắp xếp ≤ 26 số (`O(σ log σ)`), tạo `L`, `R` và nhân (`O(σ)`).
- Hai lượt, mỗi lượt `n` bước.

```text
Time  : O(n · σ log σ)  ≈  O(26 · n)   (σ là hằng số nhỏ)
Space : O(n · σ)        (right_pos giữ ≤ 27 số cho mỗi vị trí)
```

Với `n = 10^5`: cỡ vài triệu thao tác — chạy được. Bộ nhớ `right_pos` khoảng `2.7 · 10^6` tham chiếu (cỡ vài chục MB trong Python) — chấp nhận được. Ở mục LeetCode ta sẽ bỏ luôn bước `sorted` bằng mẹo "đưa lên đầu" (move-to-front); ở phần bài tập có thử thách giảm bộ nhớ xuống `O(n)`.

# 9. Mô phỏng Cách 3 trên `s = "abcac"` (n = 5)

## Lượt 1 — từ phải sang trái

| t | s[t] | `first` sau khi ghi đè | `q = right_pos[t]` (+ lính canh 5) | `R` = hiệu liên tiếp |
|---|---|---|---|---|
| 4 | c | {c: 4} | [4, 5] | [1] |
| 3 | a | {c: 4, a: 3} | [3, 4, 5] | [1, 1] |
| 2 | c | {c: 2, a: 3} | [2, 3, 5] | [1, 2] |
| 1 | b | {c: 2, a: 3, b: 1} | [1, 2, 3, 5] | [1, 1, 2] |
| 0 | a | {c: 2, a: 0, b: 1} | [0, 1, 2, 5] | [1, 1, 3] |

Để ý dòng `t = 2`: chữ `c` ở vị trí 4 bị **ghi đè** thành 2, vì trong `s[2..]` chữ `c` xuất hiện đầu tiên tại 2.

## Lượt 2 — từ trái sang phải

| j | s[j] | `last` | `p` (+ lính canh −1) | `L` | `q = right_pos[j+1]` | `R` | Σ L·R | ans |
|---|---|---|---|---|---|---|---|---|
| 0 | a | {a: 0} | [0, −1] | [1] | [1, 2, 3, 5] | [1, 1, 2] | 1·1 = 1 | 1 |
| 1 | b | {a: 0, b: 1} | [1, 0, −1] | [1, 1] | [2, 3, 5] | [1, 2] | 1·1 + 1·2 = 3 | 4 |
| 2 | c | {a: 0, b: 1, c: 2} | [2, 1, 0, −1] | [1, 1, 1] | [3, 4, 5] | [1, 1] | 1·1 + 1·1 = 2 | 6 |
| 3 | a | {a: 3, b: 1, c: 2} | [3, 2, 1, −1] | [1, 1, 2] | [4, 5] | [1] | 1·1 = 1 | 7 |

Dòng `j = 3` rất đáng chú ý: `last` theo **thứ tự chèn** là `{a: 3, b: 1, c: 2}` → các giá trị `[3, 1, 2]` **không** giảm dần. Phải `sorted` mới được `[3, 2, 1]`. Quên sắp xếp là một lỗi thật (xem mục Lỗi thường gặp).

Kết quả cuối `ans = 7`, khớp đề. Hàm `trace_breakpoints` dưới in đúng các trạng thái này.

In [20]:
def trace_breakpoints(s: str) -> int:
    n = len(s)
    print("=== Lượt 1: từ phải sang trái ===")
    right_pos = [None] * n
    first = {}
    for t in range(n - 1, -1, -1):
        first[s[t]] = t
        q = sorted(first.values()) + [n]
        right_pos[t] = q
        R = [q[d + 1] - q[d] for d in range(len(q) - 1)]
        print(f"t={t} s[t]={s[t]!r} first={first} q={q} R={R}")
    print("=== Lượt 2: từ trái sang phải ===")
    last = {}
    ans = 0
    for j in range(n - 1):
        last[s[j]] = j
        p = sorted(last.values(), reverse=True) + [-1]
        L = [p[d] - p[d + 1] for d in range(len(p) - 1)]
        q = right_pos[j + 1]
        R = [q[d + 1] - q[d] for d in range(len(q) - 1)]
        contrib = sum(a * b for a, b in zip(L, R))
        ans += contrib
        print(f"j={j} s[j]={s[j]!r} last={last} p={p} L={L} | q={q} R={R} -> +{contrib}, ans={ans}")
    return ans

result = trace_breakpoints("abcac")
print("Kết quả:", result)

=== Lượt 1: từ phải sang trái ===
t=4 s[t]='c' first={'c': 4} q=[4, 5] R=[1]
t=3 s[t]='a' first={'c': 4, 'a': 3} q=[3, 4, 5] R=[1, 1]
t=2 s[t]='c' first={'c': 2, 'a': 3} q=[2, 3, 5] R=[1, 2]
t=1 s[t]='b' first={'c': 2, 'a': 3, 'b': 1} q=[1, 2, 3, 5] R=[1, 1, 2]
t=0 s[t]='a' first={'c': 2, 'a': 0, 'b': 1} q=[0, 1, 2, 5] R=[1, 1, 3]
=== Lượt 2: từ trái sang phải ===
j=0 s[j]='a' last={'a': 0} p=[0, -1] L=[1] | q=[1, 2, 3, 5] R=[1, 1, 2] -> +1, ans=1
j=1 s[j]='b' last={'a': 0, 'b': 1} p=[1, 0, -1] L=[1, 1] | q=[2, 3, 5] R=[1, 2] -> +3, ans=4
j=2 s[j]='c' last={'a': 0, 'b': 1, 'c': 2} p=[2, 1, 0, -1] L=[1, 1, 1] | q=[3, 4, 5] R=[1, 1] -> +2, ans=6
j=3 s[j]='a' last={'a': 3, 'b': 1, 'c': 2} p=[3, 2, 1, -1] L=[1, 1, 2] | q=[4, 5] R=[1] -> +1, ans=7
Kết quả: 7


# 10. Bước đột phá 3 — "đúng d" = "nhiều nhất d" − "nhiều nhất (d − 1)"

Cách 3 tìm điểm gãy bằng vị trí xuất hiện cuối cùng. Có một con đường thứ hai, cũng dựa trên tính đơn điệu, dẫn tới một **pattern dùng được ở rất nhiều bài**.

## Suy luận

Cố định điểm cuối `r`. Định nghĩa:

```text
f_d[r] = độ dài của hậu tố DÀI NHẤT kết thúc tại r có NHIỀU NHẤT d ký tự phân biệt
f_0[r] = 0
```

Vì bậc thang không giảm (mục 3.5), các hậu tố có `≤ d` ký tự phân biệt là **đúng** các hậu tố có độ dài `1, 2, …, f_d[r]` — một khối liền nhau, không có "lỗ".

Do đó số hậu tố có **đúng** `d` ký tự phân biệt:

```text
exact_d[r] = f_d[r] − f_{d−1}[r]
```

## Ví dụ `s = "abcac"`, `r = 4`

Các hậu tố: `"c"`(1), `"ac"`(2), `"cac"`(2), `"bcac"`(3), `"abcac"`(3).

- `f_1[4] = 1` (chỉ `"c"`), `f_2[4] = 3` (tới `"cac"`), `f_3[4] = 5` (cả chuỗi).
- `exact_1 = 1 − 0 = 1`, `exact_2 = 3 − 1 = 2`, `exact_3 = 5 − 3 = 2`.

Khớp với việc đếm trực tiếp: một hậu tố có 1 chữ, hai hậu tố có 2 chữ, hai hậu tố có 3 chữ.

In [21]:
s = "abcac"
r = 4
suffixes = [s[i:r + 1] for i in range(r, -1, -1)]
dist = [len(set(x)) for x in suffixes]
print("Hậu tố:", suffixes)
print("distinct:", dist)

f = [0] + [max(len(x) for x, dx in zip(suffixes, dist) if dx <= d) for d in range(1, 4)]
print("f_0..f_3 tại r = 4:", f)
for d in range(1, 4):
    print(f"d = {d}: f_d - f_(d-1) = {f[d]} - {f[d - 1]} = {f[d] - f[d - 1]} | đếm trực tiếp = {dist.count(d)}")

Hậu tố: ['c', 'ac', 'cac', 'bcac', 'abcac']
distinct: [1, 2, 2, 3, 3]
f_0..f_3 tại r = 4: [0, 1, 3, 5]
d = 1: f_d - f_(d-1) = 1 - 0 = 1 | đếm trực tiếp = 1
d = 2: f_d - f_(d-1) = 3 - 1 = 2 | đếm trực tiếp = 2
d = 3: f_d - f_(d-1) = 5 - 3 = 2 | đếm trực tiếp = 2


## Tính `f_d[r]` cho mọi `r` bằng hai con trỏ

Đây đúng là bài toán con ở mục 3.7, chỉ khác là ta **ghi lại** độ dài cửa sổ tại mỗi `r` thay vì lấy max:

```python
for r in range(n):
    thêm s[r] vào cửa sổ
    while distinct > d:
        bỏ s[l], l += 1
    f[r] = r - l + 1
```

Vì sao kết quả đúng là "hậu tố dài nhất"? Vì `l` là vị trí **nhỏ nhất** còn giữ được `≤ d` chữ: nếu lùi `l` thêm một bước thì lúc trước đó vòng `while` đã phải bỏ nó đi, tức đoạn dài hơn có `> d` chữ. Và `l` chỉ tiến nên mỗi lần gọi là `O(n)`.

Ta dùng mảng mã số `codes` (mục 3.3) để có thể thêm/bớt chữ cái.

In [22]:
def at_most_lengths(codes, d):
    """f[r] = độ dài hậu tố dài nhất kết thúc tại r có <= d ký tự phân biệt."""
    n = len(codes)
    cnt = [0] * 26
    f = [0] * n
    l = 0
    distinct = 0
    for r in range(n):
        c = codes[r]
        if cnt[c] == 0:
            distinct += 1
        cnt[c] += 1
        while distinct > d:
            cl = codes[l]
            cnt[cl] -= 1
            if cnt[cl] == 0:
                distinct -= 1
            l += 1
        f[r] = r - l + 1
    return f

s = "abcac"
codes = [ord(ch) - ord('a') for ch in s]
print("codes =", codes)
for d in range(1, 4):
    print(f"f_{d} =", at_most_lengths(codes, d))

codes = [0, 1, 2, 0, 2]
f_1 = [1, 1, 1, 1, 1]
f_2 = [1, 2, 2, 2, 3]
f_3 = [1, 2, 3, 4, 5]


Đọc kết quả: `f_1 = [1, 1, 1, 1, 1]`, `f_2 = [1, 2, 2, 2, 3]`, `f_3 = [1, 2, 3, 4, 5]`.

Lấy cột `r = 3` (tương ứng vết cắt `j = 3`): `exact = [1 − 0, 2 − 1, 4 − 2] = [1, 1, 2]` — đúng bằng `L_3` trong bảng mục 5.

**Bên phải** dùng lại y nguyên hàm này trên chuỗi đảo `rev` (mục 3.8): các tiền tố bắt đầu tại `j + 1` của `s` ↔ các hậu tố kết thúc tại `n − 2 − j` của `rev`. Cell dưới kiểm tra ánh xạ đó cho mọi vết cắt của `"abcac"`.

In [23]:
s = "abcac"
n = len(s)
rev_codes = [ord(ch) - ord('a') for ch in s[::-1]]
f_rev = {d: at_most_lengths(rev_codes, d) for d in range(1, 4)}
f_rev[0] = [0] * n
for j in range(n - 1):
    r = n - 2 - j
    R_from_rev = [f_rev[d][r] - f_rev[d - 1][r] for d in range(1, 4)]
    _, R_direct = side_counts(s, j)
    print(f"j = {j}: r = n-2-j = {r} | R từ rev = {R_from_rev} | R trực tiếp = {R_direct[1:4]}")
    assert R_from_rev == R_direct[1:4]

j = 0: r = n-2-j = 3 | R từ rev = [1, 1, 2] | R trực tiếp = [1, 1, 2]
j = 1: r = n-2-j = 2 | R từ rev = [1, 2, 0] | R trực tiếp = [1, 2, 0]
j = 2: r = n-2-j = 1 | R từ rev = [1, 1, 0] | R trực tiếp = [1, 1, 0]
j = 3: r = n-2-j = 0 | R từ rev = [1, 0, 0] | R trực tiếp = [1, 0, 0]


# 11. Cách 4 — Cửa sổ trượt cho từng `d`: `O(26 · n)`

## Thuật toán

1. `codes` = mã số của `s`, `rev` = mã số của `s` đảo ngược, `D` = số chữ cái khác nhau trong `s` (`D <= 26`).
2. Giữ hai mảng `prev_left`, `prev_right` là `f_{d−1}` của hai phía (ban đầu toàn `0`, chính là `f_0`).
3. Với `d = 1, 2, …, D`:
   - `cur_left = at_most_lengths(codes, d)`, `cur_right = at_most_lengths(rev, d)`.
   - Với mỗi vết cắt `j`: `r = n − 2 − j`,
     `ans += (cur_left[j] − prev_left[j]) · (cur_right[r] − prev_right[r])`.
   - Gán `prev = cur` để dùng cho `d + 1`.

**Vì sao chỉ cần tới `D`?** Khi `d >= D`, mọi đoạn đều có `≤ d` chữ, nên `f_d[r] = r + 1 = f_{d+1}[r]` và hiệu bằng 0 — không đóng góp gì.

**Vì sao chỉ giữ `prev`?** Công thức `exact_d` chỉ cần `f_d` và `f_{d−1}`, nên bộ nhớ là `O(n)` thay vì lưu cả bảng `26 × n` như lời giải gốc thường làm.

In [24]:
def count_magical_sliding_window(s: str) -> int:
    n = len(s)
    codes = [ord(ch) - ord('a') for ch in s]
    rev = codes[::-1]
    D = len(set(s))
    prev_left = [0] * n          # f_{d-1} của s
    prev_right = [0] * n         # f_{d-1} của rev
    ans = 0
    for d in range(1, D + 1):
        cur_left = at_most_lengths(codes, d)
        cur_right = at_most_lengths(rev, d)
        for j in range(n - 1):
            r = n - 2 - j
            exact_left = cur_left[j] - prev_left[j]
            exact_right = cur_right[r] - prev_right[r]
            ans += exact_left * exact_right
        prev_left, prev_right = cur_left, cur_right
    return ans

for s, expected in examples:
    got = count_magical_sliding_window(s)
    print(f"{s!r:8} -> {got} (mong đợi {expected})")
    assert got == expected

'aba'    -> 2 (mong đợi 2)
'kkk'    -> 4 (mong đợi 4)
'abcac'  -> 7 (mong đợi 7)


## Độ phức tạp Cách 4

Gọi `D` = số chữ cái khác nhau của `s` (`D <= 26`).

- Mỗi giá trị `d`: hai lần `at_most_lengths` (`O(n)` mỗi lần nhờ `l` chỉ tiến) + một vòng ghép `O(n)`.
- Có `D` giá trị `d`.

```text
Time  : O(D · n) <= O(26 · n)
Space : O(n)      (bốn mảng độ dài n)
```

So với Cách 3: cùng bậc thời gian, ít bộ nhớ hơn, nhưng trong Python thường chậm hơn một chút vì vòng `while` của hai con trỏ được chạy lại 26 lần. Cái lợi lớn nhất của Cách 4 là **pattern "đúng K = nhiều nhất K − nhiều nhất K−1"** — bạn sẽ gặp lại nó ở LeetCode 992 và nhiều bài khác.

# 12. Mô phỏng hai con trỏ: `at_most_lengths` với `s = "abcac"`, `d = 2`

| r | s[r] | sau khi thêm: distinct | co cửa sổ? | cửa sổ `s[l..r]` | `f_2[r]` |
|---|---|---|---|---|---|
| 0 | a | 1 | không | [0..0] = "a" | 1 |
| 1 | b | 2 | không | [0..1] = "ab" | 2 |
| 2 | c | 3 > 2 | bỏ 'a' → l = 1, distinct = 2 | [1..2] = "bc" | 2 |
| 3 | a | 3 > 2 | bỏ 'b' → l = 2, distinct = 2 | [2..3] = "ca" | 2 |
| 4 | c | 2 (đã có 'c') | không | [2..4] = "cac" | 3 |

Quan sát: `l` đi `0 → 0 → 1 → 2 → 2`, **chỉ tiến**. Ở `r = 4`, thêm `'c'` không tạo chữ mới vì `cnt[c]` đang là 1, nên cửa sổ được dài ra thành `"cac"`.

Kết quả `f_2 = [1, 2, 2, 2, 3]`, khớp cell ở mục 10.

In [25]:
def trace_at_most(s: str, d: int):
    codes = [ord(ch) - ord('a') for ch in s]
    cnt = [0] * 26
    l = 0
    distinct = 0
    f = []
    for r, c in enumerate(codes):
        if cnt[c] == 0:
            distinct += 1
        cnt[c] += 1
        note = f"thêm {s[r]!r} -> distinct = {distinct}"
        removed = []
        while distinct > d:
            cl = codes[l]
            cnt[cl] -= 1
            if cnt[cl] == 0:
                distinct -= 1
            removed.append(s[l])
            l += 1
        if removed:
            note += f" | bỏ {removed} -> l = {l}, distinct = {distinct}"
        f.append(r - l + 1)
        print(f"r = {r}: {note:58} cửa sổ = {s[l:r + 1]!r:6} f[r] = {r - l + 1}")
    return f

print("f_2 =", trace_at_most("abcac", 2))

r = 0: thêm 'a' -> distinct = 1                                   cửa sổ = 'a'    f[r] = 1
r = 1: thêm 'b' -> distinct = 2                                   cửa sổ = 'ab'   f[r] = 2
r = 2: thêm 'c' -> distinct = 3 | bỏ ['a'] -> l = 1, distinct = 2 cửa sổ = 'bc'   f[r] = 2
r = 3: thêm 'a' -> distinct = 3 | bỏ ['b'] -> l = 2, distinct = 2 cửa sổ = 'ca'   f[r] = 2
r = 4: thêm 'c' -> distinct = 2                                   cửa sổ = 'cac'  f[r] = 3
f_2 = [1, 2, 2, 2, 3]


# 13. Phiên bản nộp LeetCode

Ta chọn **Cách 3** làm bản nộp (nhanh nhất trong Python), với hai tinh chỉnh:

1. **Đổi chữ cái sang mã số** `0..25` và dùng list 26 ô thay cho `dict`.
2. **Bỏ `sorted` bằng mẹo "đưa lên đầu" (move-to-front).** Giữ list `order` gồm các chữ cái, xếp theo vị trí xuất hiện **gần đây nhất trước**. Khi gặp chữ `c` ở vị trí `j`, `c` trở thành chữ gần đây nhất → xóa `c` khỏi `order` (nếu có) rồi chèn vào đầu. Khi đó `[last[x] for x in order]` **tự động giảm dần** — không cần sắp xếp.

`order` dài tối đa 26 nên `remove` / `insert(0, …)` chỉ tốn `O(26)`. Bên phải làm y hệt khi duyệt ngược (chữ vừa gặp là chữ có vị trí đầu tiên **nhỏ nhất**, nên cũng đưa lên đầu).

Cell dưới minh họa `order` thay đổi thế nào trên `"abcac"`.

In [26]:
def demo_move_to_front(s: str) -> None:
    order = []
    last = {}
    for j, ch in enumerate(s):
        if ch in order:
            order.remove(ch)
        order.insert(0, ch)
        last[ch] = j
        p = [last[x] for x in order]
        print(f"j = {j} gặp {ch!r}: order = {order}  -> vị trí cuối = {p}"
              f"  (giống sorted? {p == sorted(last.values(), reverse=True)})")

demo_move_to_front("abcac")

j = 0 gặp 'a': order = ['a']  -> vị trí cuối = [0]  (giống sorted? True)
j = 1 gặp 'b': order = ['b', 'a']  -> vị trí cuối = [1, 0]  (giống sorted? True)
j = 2 gặp 'c': order = ['c', 'b', 'a']  -> vị trí cuối = [2, 1, 0]  (giống sorted? True)
j = 3 gặp 'a': order = ['a', 'c', 'b']  -> vị trí cuối = [3, 2, 1]  (giống sorted? True)
j = 4 gặp 'c': order = ['c', 'a', 'b']  -> vị trí cuối = [4, 3, 1]  (giống sorted? True)


In [27]:
from typing import List

class Solution:
    def countMagicalPairs(self, s: str) -> int:
        n = len(s)
        codes = [ord(ch) - 97 for ch in s]

        # Lượt 1 (phải -> trái): right_pos[t] = vị trí xuất hiện đầu tiên trong s[t..],
        # tăng dần, kèm lính canh n.
        right_pos = [None] * n
        order = []
        seen = [False] * 26
        first = [0] * 26
        for t in range(n - 1, -1, -1):
            c = codes[t]
            if seen[c]:
                order.remove(c)
            else:
                seen[c] = True
            order.insert(0, c)
            first[c] = t
            q = [first[x] for x in order]
            q.append(n)
            right_pos[t] = q

        # Lượt 2 (trái -> phải): ghép bên trái (vị trí cuối cùng) với bên phải.
        order = []
        seen = [False] * 26
        last = [0] * 26
        ans = 0
        for j in range(n - 1):
            c = codes[j]
            if seen[c]:
                order.remove(c)
            else:
                seen[c] = True
            order.insert(0, c)
            last[c] = j
            p = [last[x] for x in order]
            p.append(-1)
            q = right_pos[j + 1]
            m = min(len(p), len(q)) - 1
            for d in range(m):
                ans += (p[d] - p[d + 1]) * (q[d + 1] - q[d])
        return ans

## Kiểm thử bản LeetCode

Ngoài 3 ví dụ của đề, ta thêm các edge case:

| Input | Kết quả mong đợi | Lý do |
|---|---|---|
| `"ab"` | 1 | `n = 2` chỉ có một bộ ba `(0, 0, 1)`, hai bên đều 1 chữ |
| `"aa"` | 1 | như trên |
| `"a" * 10` | `C(11, 3) = 165` | một chữ cái lặp lại → mọi bộ ba đều magical |
| `"abc...z"` (26 chữ khác nhau) | 169 | mỗi đoạn có distinct = độ dài, nên cần `len(s1) = len(s2)`: tổng `Σ_j min(j + 1, 25 − j)` |
| `"a" * 10^5` | `C(100001, 3) = 166666666650000` | đáp án lớn nhất, vượt 32 bit |

In [28]:
import string

sol = Solution()
tests = [
    ("aba", 2),
    ("kkk", 4),
    ("abcac", 7),
    ("ab", 1),
    ("aa", 1),
    ("a" * 10, comb(11, 3)),
    (string.ascii_lowercase, sum(min(j + 1, 25 - j) for j in range(25))),
]
for s, expected in tests:
    got = sol.countMagicalPairs(s)
    status = "PASS" if got == expected else "FAIL"
    print(f"{s[:26]!r:30} -> {got:>5} | mong đợi {expected:>5} | {status}")
    assert got == expected

big_same = "a" * 10**5
assert sol.countMagicalPairs(big_same) == comb(10**5 + 1, 3)
print("'a' * 10^5 ->", sol.countMagicalPairs(big_same), "= C(100001, 3)")

'aba'                          ->     2 | mong đợi     2 | PASS
'kkk'                          ->     4 | mong đợi     4 | PASS
'abcac'                        ->     7 | mong đợi     7 | PASS
'ab'                           ->     1 | mong đợi     1 | PASS
'aa'                           ->     1 | mong đợi     1 | PASS
'aaaaaaaaaa'                   ->   165 | mong đợi   165 | PASS
'abcdefghijklmnopqrstuvwxyz'   ->   169 | mong đợi   169 | PASS


'a' * 10^5 -> 166666666650000 = C(100001, 3)


## Thử tốc độ với `n = 10^5`

Chuỗi ngẫu nhiên 26 chữ cái là trường hợp "nặng" nhất: hầu hết vết cắt có đủ 26 mức `d` ở cả hai phía. Cell dưới chạy bản Solution, Cách 3 và Cách 4 trên cùng một chuỗi (seed cố định), kiểm tra ba kết quả bằng nhau và in thời gian. Thời gian phụ thuộc máy; trên máy thông thường mỗi cách chạy cỡ dưới 1–2 giây.

In [29]:
import random, time

rng = random.Random(2026)
big = "".join(rng.choice(string.ascii_lowercase) for _ in range(10**5))

results = {}
for name, fn in [("Solution (Cách 3 + move-to-front)", sol.countMagicalPairs),
                 ("Cách 3 (sorted)", count_magical_breakpoints),
                 ("Cách 4 (sliding window)", count_magical_sliding_window)]:
    t0 = time.perf_counter()
    results[name] = fn(big)
    print(f"{name:36} -> {results[name]}  ({time.perf_counter() - t0:.2f} s)")

assert len(set(results.values())) == 1
print("Ba cách cho cùng kết quả.")

Solution (Cách 3 + move-to-front)    -> 165693020833073  (0.43 s)


Cách 3 (sorted)                      -> 165693020833073  (0.49 s)


Cách 4 (sliding window)              -> 165693020833073  (0.94 s)
Ba cách cho cùng kết quả.


# 14. Kiểm thử chéo (cross-check) với brute force

Chạy pass trên 3 ví dụ là chưa đủ cho bài Hard. Ta sinh **ngẫu nhiên** nhiều chuỗi nhỏ (seed cố định để kết quả lặp lại được) và so mọi cách với Cách 1 — "trọng tài" hiển nhiên đúng:

- 400 chuỗi dài `2..8`, dùng 1–4 chữ cái đầu (nhiều ký tự lặp → dễ lộ lỗi).
- 100 chuỗi dài `10..60`, dùng 1–26 chữ cái, so với Cách 2 (Cách 1 quá chậm ở độ dài này).

In [30]:
rng = random.Random(4091)
fast_methods = {
    "Cách 2": count_magical_split,
    "Cách 3": count_magical_breakpoints,
    "Cách 4": count_magical_sliding_window,
    "Solution": sol.countMagicalPairs,
}

small_cases = 0
for _ in range(400):
    n = rng.randint(2, 8)
    alpha = string.ascii_lowercase[:rng.randint(1, 4)]
    s = "".join(rng.choice(alpha) for _ in range(n))
    expected = count_magical_bruteforce(s)
    for name, fn in fast_methods.items():
        assert fn(s) == expected, (name, s, fn(s), expected)
    small_cases += 1

medium_cases = 0
for _ in range(100):
    n = rng.randint(10, 60)
    alpha = string.ascii_lowercase[:rng.randint(1, 26)]
    s = "".join(rng.choice(alpha) for _ in range(n))
    expected = count_magical_split(s)
    for name in ["Cách 3", "Cách 4", "Solution"]:
        assert fast_methods[name](s) == expected, (name, s)
    medium_cases += 1

print(f"PASS: {small_cases} chuỗi nhỏ (so với brute force) + {medium_cases} chuỗi vừa (so với Cách 2)")

PASS: 400 chuỗi nhỏ (so với brute force) + 100 chuỗi vừa (so với Cách 2)


# 15. Vì sao các cách đều đúng?

## (1) Phân rã theo vết cắt — nền móng của Cách 2, 3, 4

- Mỗi bộ ba `(i, j, k)` hợp lệ ứng với **đúng một** bộ "vết cắt `j`, lựa chọn trái `i ∈ [0, j]`, lựa chọn phải `k ∈ [j+1, n−1]`" và ngược lại → không bỏ sót, không đếm trùng.
- Với `j` cố định, chia các cặp `(i, k)` theo giá trị chung `d = distinct(s1) = distinct(s2)`. Các nhóm `d` khác nhau rời nhau; trong nhóm `d`, `i` và `k` chọn độc lập → `L_j[d] · R_j[d]` cặp.

## (2) Bổ đề bậc thang — nền móng của Cách 3

Với `j` cố định, đặt `g(i) = distinct(s[i..j])`. Khi đó:

```text
g(i) = g(i + 1) + 1   nếu s[i] không xuất hiện trong s[i+1..j]   (i là vị trí cuối của chữ s[i])
g(i) = g(i + 1)       nếu ngược lại
```

Vậy `g` tăng đúng 1 tại mỗi vị trí cuối `p_1 > p_2 > … > p_m` và hằng số giữa chúng: `g(i) = d` ⇔ `p_{d+1} < i <= p_d` (với `p_{m+1} = −1`). Số `i` như vậy là `p_d − p_{d+1}`. Bên phải đối xứng với vị trí đầu tiên và lính canh `n`.

## (3) Bất biến của hai lượt duyệt

- Lượt trái → phải: **sau khi xử lý `j`**, `last[c]` = vị trí cuối của `c` trong `s[0..j]` (vì ghi đè theo `j` tăng dần).
- Lượt phải → trái: **sau khi xử lý `t`**, `first[c]` = vị trí đầu của `c` trong `s[t..]` (vì ghi đè theo `t` giảm dần).
- Với move-to-front: `order` luôn xếp các chữ theo `last` giảm dần (chữ vừa gặp có `last` lớn nhất nên đứng đầu; thứ tự các chữ còn lại không đổi vì `last` của chúng không đổi).

## (4) Bất biến của hai con trỏ — nền móng của Cách 4

Sau mỗi bước `r`, `s[l..r]` là hậu tố **dài nhất** kết thúc tại `r` có `≤ d` chữ. Nhờ bậc thang, "hậu tố có `≤ d` chữ" là một khối độ dài `1..f_d[r]`, nên `f_d[r] − f_{d−1}[r]` đếm đúng các hậu tố có **đúng** `d` chữ.

# 16. Độ phức tạp và so sánh các cách

Ký hiệu: `n = len(s)`, `σ = 26` (kích thước bảng chữ cái), `D` = số chữ cái khác nhau thực sự có trong `s` (`D <= σ`).

| Cách | Ý tưởng | Thời gian | Bộ nhớ phụ | Dùng khi nào |
|---|---|---|---|---|
| 1. Brute Force | thử mọi `(i, j, k)` + `set` | `O(n^4)` | `O(n)` | trọng tài kiểm thử, `n <= ~30` |
| 2. Theo vết cắt | kéo dài 2 phía + nguyên lý nhân | `O(n^2)` | `O(σ)` | `n` vài nghìn |
| 3. Điểm gãy (`sorted`) | vị trí cuối / đầu tiên | `O(n · σ log σ)` | `O(n · σ)` | dễ hiểu, đủ nhanh |
| 3'. Solution (move-to-front) | như Cách 3, bỏ sắp xếp | `O(n · σ)` | `O(n · σ)` | **bản nộp** |
| 4. Cửa sổ trượt | `exact = atMost(d) − atMost(d−1)` | `O(n · D)` | `O(n)` | pattern tổng quát, ít bộ nhớ |

**Ghi chú trung thực về Python:** `26 · n = 2.6 · 10^6` vòng lặp nhỏ là chấp nhận được trên LeetCode (cỡ dưới 1 giây cho bản Solution trên máy thông thường), nhưng hằng số của Python lớn. Nếu bị Time Limit Exceeded, hãy: dùng bản move-to-front (không `sorted`), chỉ nhân tới `min(len(p), len(q))`, tránh tạo nhiều list tạm, hoặc chuyển sang bản một lượt duyệt ở Bài tập 6.

# 17. Những lỗi thường gặp

## Lỗi 1 — Slicing lệch một vị trí

Sai:

```python
s1 = s[i:j]        # thiếu s[j]
s2 = s[j:k]        # lấy nhầm s[j], thiếu s[k]
```

Đề dùng `s[i..j]` **đóng hai đầu**. Đúng phải là `s[i:j+1]` và `s[j+1:k+1]`. Bản sai còn tạo ra chuỗi **rỗng** khi `i == j` (vì `s[i:i]` rỗng), làm số đếm sai hẳn.

In [31]:
def wrong_bruteforce_slicing(s: str) -> int:
    n = len(s)
    ans = 0
    for i in range(n):
        for j in range(i, n):
            for k in range(j + 1, n):
                if len(set(s[i:j])) == len(set(s[j:k])):     # SAI: lệch chỉ số
                    ans += 1
    return ans

for s, expected in examples:
    print(f"{s!r:8} bản sai = {wrong_bruteforce_slicing(s)} | đúng = {expected}")

'aba'    bản sai = 1 | đúng = 2
'kkk'    bản sai = 1 | đúng = 4
'abcac'  bản sai = 4 | đúng = 7


## Lỗi 2 — Lính canh sai giá trị

Sai: dùng `p.append(0)` cho bên trái (nghĩ "chỉ số nhỏ nhất là 0") hoặc `q.append(n - 1)` cho bên phải.

Vì sao sai? Bậc cuối của bên trái gồm các `i = p_m, p_m − 1, …, 0` — có `p_m − (−1) = p_m + 1` giá trị. Dùng `0` làm lính canh sẽ **mất đúng một** `i` (chính là `i = 0`). Tương tự bên phải phải dùng `n`.

## Lỗi 3 — Quên sắp xếp, tin vào thứ tự của `dict`

Sai: `p = list(last.values())`. Như bảng mô phỏng ở `j = 3` của `"abcac"`, `last = {a: 3, b: 1, c: 2}` cho `[3, 1, 2]` — không giảm dần, các hiệu sẽ âm hoặc sai.

Cell dưới cài một bản Cách 3 có "công tắc" để bật từng lỗi và xem kết quả lệch ra sao.

In [32]:
def breakpoints_with_bug(s: str, left_sentinel=-1, right_sentinel=None, use_sort=True) -> int:
    n = len(s)
    if right_sentinel is None:
        right_sentinel = n
    right_pos = [None] * n
    first = {}
    for t in range(n - 1, -1, -1):
        first[s[t]] = t
        q = sorted(first.values()) if use_sort else list(first.values())
        right_pos[t] = q + [right_sentinel]
    last = {}
    ans = 0
    for j in range(n - 1):
        last[s[j]] = j
        p = sorted(last.values(), reverse=True) if use_sort else list(last.values())
        p = p + [left_sentinel]
        q = right_pos[j + 1]
        L = [p[d] - p[d + 1] for d in range(len(p) - 1)]
        R = [q[d + 1] - q[d] for d in range(len(q) - 1)]
        ans += sum(a * b for a, b in zip(L, R))
    return ans

for s, expected in examples:
    print(f"{s!r:8} đúng = {expected} | lính canh trái 0: {breakpoints_with_bug(s, left_sentinel=0)}"
          f" | lính canh phải n-1: {breakpoints_with_bug(s, right_sentinel=len(s) - 1)}"
          f" | quên sorted: {breakpoints_with_bug(s, use_sort=False)}")

'aba'    đúng = 2 | lính canh trái 0: 1 | lính canh phải n-1: 1 | quên sorted: -2
'kkk'    đúng = 4 | lính canh trái 0: 1 | lính canh phải n-1: 1 | quên sorted: 4
'abcac'  đúng = 7 | lính canh trái 0: 4 | lính canh phải n-1: 4 | quên sorted: 5


Đọc kết quả:

- Hai lỗi lính canh làm lệch **mọi** ví dụ (mỗi vết cắt mất đúng một lựa chọn ở bậc cuối).
- Lỗi "quên sorted" thì tinh vi hơn: `"kkk"` vẫn ra đúng `4` (chỉ có một chữ cái nên chẳng có gì để sắp xếp); `"aba"` ra `-2` — một số **âm**, dấu hiệu rõ ràng của hiệu âm; còn `"abcac"` ra `5` — sai nhưng trông "hợp lý", rất dễ bị bỏ qua.

Đó là lý do phải **kiểm thử chéo ngẫu nhiên** chứ không chỉ chạy ví dụ của đề.

## Lỗi 4 — Nhân số đếm "nhiều nhất" thay vì "đúng"

Sai: `ans += cur_left[j] * cur_right[r]` (quên trừ `prev`). `f_d` đếm các đoạn có **≤ d** chữ — một đoạn 1 chữ sẽ bị tính ở cả `d = 1, 2, 3, …` → đếm trùng.

## Lỗi 5 — Ánh xạ chỉ số đảo ngược sai

Sai: `r = n - 1 - j`. Vị trí `j` của `s` ứng với `n − 1 − j` trong `rev`, nhưng bên phải bắt đầu tại **`j + 1`**, nên phải là `r = n − 1 − (j + 1) = n − 2 − j`.

In [33]:
def window_with_bug(s: str, mode: str) -> int:
    n = len(s)
    codes = [ord(ch) - ord('a') for ch in s]
    rev = codes[::-1]
    prev_left, prev_right = [0] * n, [0] * n
    ans = 0
    for d in range(1, len(set(s)) + 1):
        cur_left, cur_right = at_most_lengths(codes, d), at_most_lengths(rev, d)
        for j in range(n - 1):
            if mode == "quên trừ":
                ans += cur_left[j] * cur_right[n - 2 - j]
            elif mode == "sai chỉ số":
                r = n - 1 - j
                ans += (cur_left[j] - prev_left[j]) * (cur_right[r] - prev_right[r])
        prev_left, prev_right = cur_left, cur_right
    return ans

for s, expected in examples:
    print(f"{s!r:8} đúng = {expected} | quên trừ prev: {window_with_bug(s, 'quên trừ')}"
          f" | r = n-1-j: {window_with_bug(s, 'sai chỉ số')}")

'aba'    đúng = 2 | quên trừ prev: 6 | r = n-1-j: 3
'kkk'    đúng = 4 | quên trừ prev: 4 | r = n-1-j: 7
'abcac'  đúng = 7 | quên trừ prev: 38 | r = n-1-j: 8


Đọc kết quả: cả hai lỗi đều lộ trên `"aba"` và `"abcac"`. Riêng `"kkk"` với bản "quên trừ" vẫn ra đúng `4`, vì chuỗi chỉ có 1 chữ cái nên chỉ có mức `d = 1` (khi đó "nhiều nhất 1" trùng với "đúng 1"). Lại thêm một ví dụ cho thấy test dễ không đủ để bắt lỗi.

## Lỗi 6 — Cho vết cắt `j` chạy tới `n − 1`

Sai: `for j in range(n):`. Khi `j = n − 1` thì không còn chỗ cho `s2` (cần `k >= j + 1 = n`, vượt chuỗi). Ở Cách 3 còn gây `IndexError` vì truy cập `right_pos[n]`. Đúng: `for j in range(n - 1)`.

## Lỗi 7 — Tràn số ở ngôn ngữ khác

Đáp án có thể lên tới `166666666650000` (với `n = 10^5`) — vượt xa `2^31 − 1 ≈ 2.1 · 10^9`. Python không sao, nhưng C++/Java phải dùng `long long` / `long` cho cả biến `ans` lẫn **tích** `L · R` (mỗi thừa số có thể tới `10^5`, tích tới `10^10`).

## Lỗi 8 — Hai con trỏ nhưng "reset" `l` mỗi vòng

Nếu viết `l = 0` lại cho mỗi `r` rồi tìm từ đầu, độ phức tạp thành `O(n^2)` cho mỗi `d`. Sức mạnh của sliding window nằm ở chỗ `l` **chỉ tiến**.

## Edge case cần nhớ

- `n = 2`: luôn trả về `1`.
- Toàn một chữ cái: đáp án `C(n + 1, 3)`, lớn nhất có thể.
- Đủ 26 chữ khác nhau đứng cạnh nhau: distinct = độ dài, nên chỉ cần `len(s1) = len(s2)`.
- Chuỗi có chữ cái xen kẽ lặp lại (như `"abcac"`): nơi lộ lỗi "quên sorted".

# 18. Mẹo nhận dạng và bài liên quan

## Dấu hiệu nhận dạng

| Bạn thấy trong đề… | Hãy nghĩ tới… |
|---|---|
| "bộ ba `i <= j < k`", hai đoạn **kề nhau** | cố định **vết cắt** `j`, tách thành bên trái × bên phải |
| điều kiện chỉ so sánh hai **con số** (distinct, tổng, max…) | đếm theo giá trị rồi dùng **nguyên lý nhân** |
| "số ký tự phân biệt" + "chữ cái thường" | chỉ có **≤ 26 mức** → duyệt theo `d = 1..26` |
| đại lượng **đơn điệu** khi kéo dài đoạn | **bậc thang**, điểm gãy, hai con trỏ |
| "**đúng** K phần tử phân biệt" | `exact(K) = atMost(K) − atMost(K − 1)` |
| cần thông tin "lần cuối gặp chữ c" | `last[c] = i` ghi đè khi duyệt; move-to-front |
| cần thông tin cho đoạn bắt đầu tại `j + 1` | **đảo chuỗi** hoặc duyệt phải → trái |

## Bài liên quan

| Bài | Liên hệ |
|---|---|
| LeetCode 1525 — Number of Good Ways to Split a String | bản "một vết cắt, dùng cả hai phía": đếm vết cắt mà distinct(trái) = distinct(phải) |
| LeetCode 992 — Subarrays with K Different Integers | đúng pattern `atMost(K) − atMost(K − 1)` |
| LeetCode 340 / 159 — Longest Substring with At Most K / Two Distinct Characters | hàm `at_most_lengths` chính là bài này |
| LeetCode 3 — Longest Substring Without Repeating Characters | hai con trỏ cơ bản |
| LeetCode 2262 — Total Appeal of a String | tổng distinct của mọi substring, dùng vị trí xuất hiện cuối cùng |
| LeetCode 828 — Count Unique Characters of All Substrings | đóng góp theo vị trí xuất hiện trước / sau |
| Count Magical Substring Pairs I (cùng cuộc thi) | phiên bản I; nếu ràng buộc của nó nhỏ (vài nghìn) thì Cách 2 là đủ — hãy đọc kỹ constraints |

Hai cell dưới là "mini code" cho hai biến thể gần nhất.

In [34]:
# Biến thể A — LeetCode 1525: đếm vết cắt c (1 <= c < n) mà distinct(s[:c]) == distinct(s[c:])
def good_splits(s: str) -> int:
    n = len(s)
    prefix_distinct = [0] * n          # prefix_distinct[c] = distinct(s[:c+1])
    seen = set()
    for i, ch in enumerate(s):
        seen.add(ch)
        prefix_distinct[i] = len(seen)
    suffix_distinct = [0] * (n + 1)    # suffix_distinct[c] = distinct(s[c:])
    seen = set()
    for i in range(n - 1, -1, -1):
        seen.add(s[i])
        suffix_distinct[i] = len(seen)
    return sum(1 for c in range(1, n) if prefix_distinct[c - 1] == suffix_distinct[c])

print(good_splits("aacaba"), good_splits("abcd"))
assert good_splits("aacaba") == 2 and good_splits("abcd") == 1

2 1


In [35]:
# Biến thể B — đếm số substring có ĐÚNG K ký tự phân biệt (bản chuỗi của LeetCode 992)
def count_substrings_exactly_k(s: str, K: int) -> int:
    codes = [ord(ch) - ord('a') for ch in s]
    f_k = at_most_lengths(codes, K)
    f_k1 = at_most_lengths(codes, K - 1) if K > 1 else [0] * len(s)
    return sum(a - b for a, b in zip(f_k, f_k1))

for K in (1, 2, 3):
    fast = count_substrings_exactly_k("abcac", K)
    slow = sum(1 for i in range(5) for j in range(i, 5) if len(set("abcac"[i:j + 1])) == K)
    print(f"'abcac', K = {K}: {fast} (kiểm bằng brute: {slow})")
    assert fast == slow

'abcac', K = 1: 5 (kiểm bằng brute: 5)
'abcac', K = 2: 5 (kiểm bằng brute: 5)
'abcac', K = 3: 5 (kiểm bằng brute: 5)


# 19. Bài tập vận dụng

Sáu bài, khó dần. Hãy tự làm trước, chỉ xem **Gợi ý** khi bí, và chỉ mở **Đáp án tham khảo** ở cuối notebook sau khi đã thử.

## Bài 1 — Đoán output (không chạy code)

```python
s = "abacb"
j = 4
print([len(set(s[i:j + 1])) for i in range(j, -1, -1)])

last = {}
for i in range(j + 1):
    last[s[i]] = i
p = sorted(last.values(), reverse=True) + [-1]
print(p)
print([p[d] - p[d + 1] for d in range(len(p) - 1)])
```

Hãy ghi ra 3 dòng output, và giải thích dòng thứ ba có ý nghĩa gì với vết cắt `j = 4`.

**Gợi ý 1 (hướng tư duy):** dòng đầu là bậc thang khi `i` lùi từ 4 về 0.

**Gợi ý 2 (cấu trúc):** `last` giữ vị trí **cuối cùng** của mỗi chữ trong `s[0..4]`; tìm vị trí cuối của `a`, `b`, `c`.

**Gợi ý 3 (gần đáp án):** `last = {a: 2, b: 4, c: 3}`; dòng ba là các hiệu `p_d − p_{d+1}` — tổng của chúng phải bằng `j + 1 = 5`.

## Bài 2 — Tính bằng tay

Với `s = "abab"`, hãy tự lập bảng như mục 5:

| j | trái `s[0..j]` | `L_j` (d = 1, 2) | phải `s[j+1..]` | `R_j` (d = 1, 2) | Σ L·R |
|---|---|---|---|---|---|
| 0 | ? | ? | ? | ? | ? |
| 1 | ? | ? | ? | ? | ? |
| 2 | ? | ? | ? | ? | ? |

rồi suy ra đáp án của bài. Sau đó kiểm tra bằng `count_magical_bruteforce("abab")`.

**Gợi ý 1:** có 3 vết cắt `j = 0, 1, 2`.

**Gợi ý 2:** với `j = 2`, bên trái `"aba"` có các hậu tố `"a"`, `"ba"`, `"aba"`.

**Gợi ý 3:** đóng góp của các vết cắt lần lượt là `1`, `2`, `1`.

## Bài 3 — Điền chỗ trống: đếm bên trái bằng điểm gãy

Hoàn thiện hàm trả về list `L` mà `L[d − 1]` = số `i ∈ [0, j]` để `s[i..j]` có đúng `d` ký tự phân biệt (list dài đúng bằng số chữ cái khác nhau của `s[0..j]`). **Không** được duyệt từng `i` rồi dùng `set` — hãy dùng vị trí xuất hiện cuối cùng.

In [36]:
def exercise_3_left_counts(s: str, j: int) -> list:
    # TODO 1: tạo dict last, duyệt i từ 0 đến j, ghi đè last[s[i]] = i
    # TODO 2: p = các giá trị của last, sắp xếp GIẢM dần
    # TODO 3: thêm lính canh (giá trị nào?) vào cuối p
    # TODO 4: trả về list các hiệu p[d] - p[d + 1]
    pass


# Tự làm xong rồi mới mở test:
# assert exercise_3_left_counts("abcac", 3) == [1, 1, 2]
# assert exercise_3_left_counts("dabcbbac", 7) == [1, 1, 5, 1]
# assert exercise_3_left_counts("abacb", 4) == [1, 1, 3]

**Gợi ý 1 (hướng tư duy):** bậc thang nhảy tại vị trí cuối cùng của từng chữ; khoảng giữa hai điểm gãy liên tiếp là số `i` cho cùng một mức `d`.

**Gợi ý 2 (cấu trúc):** `sorted(last.values(), reverse=True)`; lính canh là `-1` vì `i` được xuống tới `0`.

**Gợi ý 3 (gần code):**

```python
p = sorted(last.values(), reverse=True) + [-1]
return [p[d] - p[d + 1] for d in range(len(p) - 1)]
```

## Bài 4 — Tự viết lại: hai con trỏ "nhiều nhất d"

Viết lại (không nhìn mục 10) hàm nhận **chuỗi** `s` và số `d`, trả về list `f` mà `f[r]` = độ dài hậu tố dài nhất kết thúc tại `r` có nhiều nhất `d` ký tự phân biệt.

In [37]:
def exercise_4_at_most_lengths(s: str, d: int) -> list:
    # TODO: mảng đếm 26 ô, biến distinct, con trỏ l chỉ tiến
    raise NotImplementedError("Hãy tự viết hàm này trước khi xem đáp án.")


# Gợi ý test:
# assert exercise_4_at_most_lengths("abcac", 2) == [1, 2, 2, 2, 3]
# assert exercise_4_at_most_lengths("abcac", 1) == [1, 1, 1, 1, 1]
# assert exercise_4_at_most_lengths("eceba", 2) == [1, 2, 3, 2, 2]

**Gợi ý 1:** mỗi bước: thêm `s[r]`, rồi co từ bên trái cho tới khi hợp lệ, rồi ghi `r − l + 1`.

**Gợi ý 2:** khi thêm: `if cnt[c] == 0: distinct += 1`; khi bớt: `cnt[c] -= 1; if cnt[c] == 0: distinct -= 1`.

**Gợi ý 3:** khung vòng lặp:

```python
for r in range(n):
    # thêm s[r]
    while distinct > d:
        # bớt s[l]
        l += 1
    f[r] = r - l + 1
```

## Bài 5 — Biến thể: `distinct(s1) > distinct(s2)`

Đếm số bộ ba `(i, j, k)` với `0 <= i <= j < k < n` sao cho `s1 = s[i..j]` có **nhiều hơn hẳn** ký tự phân biệt so với `s2 = s[j+1..k]`. Yêu cầu `O(26 · n)` (hoặc `O(n · 26 log 26)`).

Kết quả mong đợi: `"aba"` → 1, `"kkk"` → 0, `"abcac"` → 7, `"abab"` → 3.

In [38]:
def exercise_5_count_greater(s: str) -> int:
    # TODO: dùng lại khung Cách 3 (L và R cho mỗi vết cắt j)
    # TODO: thay Σ L[d] * R[d] bằng Σ L[d] * (tổng R của các mức nhỏ hơn d)
    pass


# Gợi ý test:
# assert exercise_5_count_greater("aba") == 1
# assert exercise_5_count_greater("kkk") == 0
# assert exercise_5_count_greater("abcac") == 7
# assert exercise_5_count_greater("abab") == 3

**Gợi ý 1 (hướng tư duy):** vẫn cố định vết cắt; bây giờ cặp hợp lệ là "mức trái `d`, mức phải `e < d`".

**Gợi ý 2 (cấu trúc):** với mỗi `j`, tính **tổng tiền tố** (prefix sum) của `R`: `acc[x] = R[0] + … + R[x − 1]`. Khi đó số `s2` có ít hơn `d + 1` chữ (với chỉ số 0-based `d`) là `acc[min(d, len(R))]`.

**Gợi ý 3 (gần code):**

```python
acc = [0]
for x in R:
    acc.append(acc[-1] + x)
for d in range(len(L)):
    ans += L[d] * acc[min(d, len(R))]
```

## Bài 6 — Mini challenge: một lượt duyệt, bộ nhớ `O(n)`

Cách 3 lưu `right_pos` cho **mọi** vị trí (`O(26 · n)` bộ nhớ). Hãy viết bản chỉ duyệt **một lượt từ trái sang phải** và chỉ dùng `O(n)` bộ nhớ phụ.

Ý tưởng: giữ list `q` = các vị trí xuất hiện đầu tiên (tăng dần) của các chữ trong **bên phải hiện tại** `s[j+1..]`. Khi vết cắt dịch từ `j` sang `j + 1`, chỉ có **một** thứ thay đổi ở bên phải: vị trí `j + 1` rời đi.

In [39]:
def exercise_6_one_pass(s: str) -> int:
    # TODO 1: tính nxt[i] = vị trí tiếp theo > i có cùng chữ với s[i] (hoặc n nếu không có)
    # TODO 2: khởi tạo q = sorted(vị trí đầu tiên của mỗi chữ trong s[1..])
    # TODO 3: với mỗi j = 0..n-2:
    #         - cập nhật bên trái (last / move-to-front), tạo p
    #         - cộng Σ (p[d] - p[d+1]) * (Q[d+1] - Q[d]) với Q = q + [n]
    #         - dịch vết cắt: bỏ q[0] (chính là j + 1), chèn nxt[j + 1] nếu < n
    pass


# Gợi ý test:
# for s, expected in examples:
#     assert exercise_6_one_pass(s) == expected

**Gợi ý 1 (hướng tư duy):** khi vị trí `j + 1` (chữ `c = s[j+1]`) rời khỏi bên phải, vị trí đầu tiên mới của chữ `c` là lần xuất hiện **kế tiếp** của `c` sau `j + 1`; mọi chữ khác giữ nguyên.

**Gợi ý 2 (cấu trúc):** tính `nxt` bằng một lượt duyệt ngược với mảng `pos[26]` ("lần gần nhất bên phải của mỗi chữ"). Vị trí `j + 1` luôn là phần tử nhỏ nhất của `q`, nên `q.pop(0)` xóa đúng nó.

**Gợi ý 3 (gần code):**

```python
q.pop(0)                      # bỏ vị trí j + 1
if nxt[j + 1] < n:
    bisect.insort(q, nxt[j + 1])
```

# Đáp án tham khảo

Bạn nên tự làm trước khi mở phần này.

---

## Đáp án Bài 1

Output:

```text
[1, 2, 3, 3, 3]
[4, 3, 2, -1]
[1, 1, 3]
```

Ý nghĩa dòng ba: với vết cắt `j = 4`, có 1 hậu tố có 1 chữ phân biệt (`"b"`), 1 hậu tố có 2 chữ (`"cb"`), và 3 hậu tố có 3 chữ (`"acb"`, `"bacb"`, `"abacb"`) — tổng 5 = `j + 1`. Cell dưới chạy lại đúng đoạn code của đề bài tập để kiểm chứng.

In [40]:
s = "abacb"
j = 4
line1 = [len(set(s[i:j + 1])) for i in range(j, -1, -1)]
last = {}
for i in range(j + 1):
    last[s[i]] = i
p = sorted(last.values(), reverse=True) + [-1]
line3 = [p[d] - p[d + 1] for d in range(len(p) - 1)]
print(line1)
print(p)
print(line3)
assert line1 == [1, 2, 3, 3, 3] and p == [4, 3, 2, -1] and line3 == [1, 1, 3]

[1, 2, 3, 3, 3]
[4, 3, 2, -1]
[1, 1, 3]


## Đáp án Bài 2

| j | trái `s[0..j]` | `L_j` (d = 1, 2) | phải `s[j+1..]` | `R_j` (d = 1, 2) | Σ L·R |
|---|---|---|---|---|---|
| 0 | "a" | [1, 0] | "bab" | [1, 2] | 1·1 = 1 |
| 1 | "ab" | [1, 1] | "ab" | [1, 1] | 1·1 + 1·1 = 2 |
| 2 | "aba" | [1, 2] | "b" | [1, 0] | 1·1 = 1 |

Đáp án: `1 + 2 + 1 = 4`.

In [41]:
s = "abab"
total = 0
for j in range(len(s) - 1):
    L, R = side_counts(s, j)
    contrib = sum(L[d] * R[d] for d in range(1, 27))
    total += contrib
    print(f"j = {j}: L[1..2] = {L[1:3]} R[1..2] = {R[1:3]} -> {contrib}")
print("Tổng =", total)
assert total == 4 == count_magical_bruteforce("abab")

j = 0: L[1..2] = [1, 0] R[1..2] = [1, 2] -> 1
j = 1: L[1..2] = [1, 1] R[1..2] = [1, 1] -> 2
j = 2: L[1..2] = [1, 2] R[1..2] = [1, 0] -> 1
Tổng = 4


## Đáp án Bài 3

In [42]:
def answer_3_left_counts(s: str, j: int) -> list:
    last = {}
    for i in range(j + 1):
        last[s[i]] = i
    p = sorted(last.values(), reverse=True) + [-1]
    return [p[d] - p[d + 1] for d in range(len(p) - 1)]

assert answer_3_left_counts("abcac", 3) == [1, 1, 2]
assert answer_3_left_counts("dabcbbac", 7) == [1, 1, 5, 1]
assert answer_3_left_counts("abacb", 4) == [1, 1, 3]

# So với đếm trực tiếp trên nhiều chuỗi ngẫu nhiên
rng = random.Random(3)
for _ in range(200):
    s = "".join(rng.choice("abcd") for _ in range(rng.randint(1, 10)))
    j = rng.randrange(len(s))
    seq = [len(set(s[i:j + 1])) for i in range(j + 1)]
    assert answer_3_left_counts(s, j) == [seq.count(d) for d in range(1, max(seq) + 1)]
print("Đáp án Bài 3: PASS")

Đáp án Bài 3: PASS


## Đáp án Bài 4

In [43]:
def answer_4_at_most_lengths(s: str, d: int) -> list:
    n = len(s)
    cnt = [0] * 26
    f = [0] * n
    l = 0
    distinct = 0
    for r in range(n):
        c = ord(s[r]) - ord('a')
        if cnt[c] == 0:
            distinct += 1
        cnt[c] += 1
        while distinct > d:
            cl = ord(s[l]) - ord('a')
            cnt[cl] -= 1
            if cnt[cl] == 0:
                distinct -= 1
            l += 1
        f[r] = r - l + 1
    return f

assert answer_4_at_most_lengths("abcac", 2) == [1, 2, 2, 2, 3]
assert answer_4_at_most_lengths("abcac", 1) == [1, 1, 1, 1, 1]
assert answer_4_at_most_lengths("eceba", 2) == [1, 2, 3, 2, 2]
print("Đáp án Bài 4: PASS", answer_4_at_most_lengths("eceba", 2))

Đáp án Bài 4: PASS [1, 2, 3, 2, 2]


## Đáp án Bài 5

Giữ nguyên hai lượt của Cách 3, chỉ đổi phép ghép: mức trái `d` (0-based) được ghép với **mọi** mức phải nhỏ hơn, lấy nhanh bằng tổng tiền tố `acc`.

In [44]:
def answer_5_count_greater(s: str) -> int:
    n = len(s)
    right_pos = [None] * n
    first = {}
    for t in range(n - 1, -1, -1):
        first[s[t]] = t
        right_pos[t] = sorted(first.values()) + [n]
    last = {}
    ans = 0
    for j in range(n - 1):
        last[s[j]] = j
        p = sorted(last.values(), reverse=True) + [-1]
        q = right_pos[j + 1]
        L = [p[d] - p[d + 1] for d in range(len(p) - 1)]
        R = [q[d + 1] - q[d] for d in range(len(q) - 1)]
        acc = [0]
        for x in R:
            acc.append(acc[-1] + x)
        for d in range(len(L)):
            ans += L[d] * acc[min(d, len(R))]
    return ans

def count_greater_bruteforce(s: str) -> int:
    n = len(s)
    return sum(1 for i in range(n) for j in range(i, n) for k in range(j + 1, n)
               if len(set(s[i:j + 1])) > len(set(s[j + 1:k + 1])))

for s, expected in [("aba", 1), ("kkk", 0), ("abcac", 7), ("abab", 3)]:
    assert answer_5_count_greater(s) == expected == count_greater_bruteforce(s)

rng = random.Random(5)
for _ in range(200):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(2, 8)))
    assert answer_5_count_greater(s) == count_greater_bruteforce(s), s
print("Đáp án Bài 5: PASS")

Đáp án Bài 5: PASS


## Đáp án Bài 6

Một lượt duyệt, bộ nhớ phụ `O(n)` (mảng `nxt`) cộng `O(26)` cho `q`, `order`. Mỗi bước: `pop(0)` và `insort` trên list ≤ 26 phần tử → vẫn `O(26 · n)` thời gian.

In [45]:
def answer_6_one_pass(s: str) -> int:
    n = len(s)
    codes = [ord(ch) - 97 for ch in s]
    # nxt[i] = vị trí tiếp theo có cùng chữ với s[i], hoặc n
    nxt = [n] * n
    pos = [n] * 26
    for i in range(n - 1, -1, -1):
        nxt[i] = pos[codes[i]]
        pos[codes[i]] = i
    # q = vị trí xuất hiện đầu tiên của từng chữ trong s[1..], tăng dần
    first_in_right = [n] * 26
    for i in range(n - 1, 0, -1):
        first_in_right[codes[i]] = i
    q = sorted(x for x in first_in_right if x < n)

    order, seen, last = [], [False] * 26, [0] * 26
    ans = 0
    for j in range(n - 1):
        c = codes[j]
        if seen[c]:
            order.remove(c)
        else:
            seen[c] = True
        order.insert(0, c)
        last[c] = j
        p = [last[x] for x in order] + [-1]
        Q = q + [n]
        for d in range(min(len(p), len(Q)) - 1):
            ans += (p[d] - p[d + 1]) * (Q[d + 1] - Q[d])
        # dịch vết cắt sang j + 1: vị trí j + 1 rời khỏi bên phải
        q.pop(0)
        if nxt[j + 1] < n:
            bisect.insort(q, nxt[j + 1])
    return ans

for s, expected in examples:
    assert answer_6_one_pass(s) == expected
rng = random.Random(6)
for _ in range(300):
    s = "".join(rng.choice(string.ascii_lowercase[:rng.randint(1, 6)]) for _ in range(rng.randint(2, 40)))
    assert answer_6_one_pass(s) == sol.countMagicalPairs(s), s
assert answer_6_one_pass(big) == results["Solution (Cách 3 + move-to-front)"]
print("Đáp án Bài 6: PASS (kể cả chuỗi ngẫu nhiên n = 10^5)")

Đáp án Bài 6: PASS (kể cả chuỗi ngẫu nhiên n = 10^5)


# 20. Tổng kết

## Ba bước đột phá cần nhớ

1. **Cố định vết cắt `j`** → bên trái và bên phải độc lập → `Σ_d L_j[d] · R_j[d]` (nguyên lý nhân + cộng).
2. **Bậc thang ≤ 26 bậc** → số ký tự phân biệt của các hậu tố chỉ nhảy tại vị trí **xuất hiện cuối cùng**; `L_j[d] = p_d − p_{d+1}` (lính canh `−1`), bên phải `R[d] = q_{d+1} − q_d` (lính canh `n`).
3. **Đúng d = nhiều nhất d − nhiều nhất (d − 1)**, mỗi "nhiều nhất d" tính bằng hai con trỏ `O(n)`.

## Template ngắn dễ nhớ (Cách 3)

```python
# Lượt 1: phải -> trái
first = {}
for t in range(n - 1, -1, -1):
    first[s[t]] = t
    right_pos[t] = sorted(first.values()) + [n]

# Lượt 2: trái -> phải
last = {}
for j in range(n - 1):
    last[s[j]] = j
    p = sorted(last.values(), reverse=True) + [-1]
    q = right_pos[j + 1]
    L = [p[d] - p[d + 1] for d in range(len(p) - 1)]
    R = [q[d + 1] - q[d] for d in range(len(q) - 1)]
    ans += sum(a * b for a, b in zip(L, R))
```

## Template "đúng K" (Cách 4)

```python
exact_K[r] = at_most(K)[r] - at_most(K - 1)[r]
```

## Checklist trước khi submit

- [ ] Slicing đúng: `s[i..j]` ↔ `s[i:j+1]`, `s[j+1..k]` ↔ `s[j+1:k+1]`.
- [ ] Vết cắt chạy `for j in range(n - 1)` (không tới `n − 1`).
- [ ] Lính canh: bên trái `−1`, bên phải `n`.
- [ ] Đã **sắp xếp** vị trí (hoặc dùng move-to-front), không tin thứ tự của `dict`.
- [ ] Ở Cách 4: nhân số đếm **đúng** `d` (đã trừ `prev`), chỉ số đảo ngược là `n − 2 − j`.
- [ ] Hai con trỏ: `l` chỉ tiến, không reset.
- [ ] Đáp án có thể tới `~1.67 · 10^14` → ngôn ngữ khác dùng `long long`.
- [ ] Đã chạy edge case: `n = 2`, toàn một chữ, đủ 26 chữ khác nhau, và **cross-check ngẫu nhiên** với brute force.
- [ ] Đã thử `n = 10^5` để chắc không quá thời gian.